In [ ]:
import os
import glob
import numpy as np
import xarray as xr
import pandas as pd

# ============================================================
# 1. 配置路径
# ============================================================
# 🚨 修正为真正的 Hindcast 数据路径
HINDCAST_O3_DIR = "/mnt/soclim0/public_data/weiji/Hindcast/0008-01/O3"
OUT_DIR = "/home/weiji/restart_exam/code/20260208HINDCAST_SUPPLY/result/0008-01/O3_eval"
os.makedirs(OUT_DIR, exist_ok=True)

# BWCN 参考数据
REF_FILE = "/mnt/soclim0/public_data/weiji/BWCN/O3/BWCN.cam.h3.0008.O3.nc"

TXT_OUT = os.path.join(OUT_DIR, "Hindcast_0008-01_O3_RMSE_Ranking.txt")
CSV_OUT = os.path.join(OUT_DIR, "Hindcast_0008-01_O3_RMSE_Ranking.csv")

LAT_BOUNDS = (60.0, 90.0)
P_TOP_HPA = 30.0
P_BOT_HPA = 70.0

# ============================================================
# 2. 核心计算逻辑 (直接沿用你提供的精确混合层积分)
# ============================================================
def calc_parc_o3_hybrid(ds_sub, p_top_hpa, p_bot_hpa):
    """混合层 O3 柱含量积分逻辑"""
    g, M_air, Na = 9.80665, 28.964 / 1000.0, 6.02214e23
    factor = Na / (g * M_air * 2.687e20)
    P0, PS = ds_sub['P0'], ds_sub['PS']
    P_interface = ds_sub['hyai'] * P0 + ds_sub['hybi'] * PS
    
    p_i = P_interface.isel(ilev=slice(0, -1)).rename({'ilev': 'lev'}).assign_coords(lev=ds_sub['lev'])
    p_ip1 = P_interface.isel(ilev=slice(1, None)).rename({'ilev': 'lev'}).assign_coords(lev=ds_sub['lev'])
    
    p_layer_top = xr.where(p_i < p_ip1, p_i, p_ip1)
    p_layer_bot = xr.where(p_i > p_ip1, p_i, p_ip1)
    
    pT, pB = p_top_hpa * 100.0, p_bot_hpa * 100.0
    upper = xr.where(p_layer_top > pT, p_layer_top, pT)
    lower = xr.where(p_layer_bot < pB, p_layer_bot, pB)
    
    overlap = xr.where(lower > upper, lower - upper, 0.0)
    return (ds_sub['O3'] * overlap * factor).sum(dim='lev')

def get_polar_mean_jan_to_may(file_path):
    """计算单文件的 60-90N 极区平均 O3 (DU)，并截取 1月1日 - 5月30日"""
    ds = xr.open_dataset(file_path)
    
    # 截取 1月到5月30日的时间
    mask_time = (ds.time.dt.month >= 1) & (ds.time.dt.month <= 5)
    mask_time = mask_time & ~((ds.time.dt.month == 5) & (ds.time.dt.day == 31))
    ds_sub = ds.isel(time=mask_time)
    
    # 调用 Hybrid 积分计算 Partial Column (保留 time, lat, lon 维度)
    o3_col = calc_parc_o3_hybrid(ds_sub, P_TOP_HPA, P_BOT_HPA)
    
    # 截取 60-90N 并进行面积加权平均
    o3_nh = o3_col.sel(lat=slice(LAT_BOUNDS[0], LAT_BOUNDS[1]))
    weights = np.cos(np.deg2rad(o3_nh.lat))
    o3_polar_mean = o3_nh.weighted(weights).mean(dim=["lat", "lon"])
    
    o3_series = o3_polar_mean.compute()
    ds.close()
    return o3_series

# ============================================================
# 3. 执行评估与排序
# ============================================================
print("[INFO] 正在计算参考数据 (BWCN) 的极区 O3 ...")
ref_o3 = get_polar_mean_jan_to_may(REF_FILE)

hindcast_files = sorted(glob.glob(os.path.join(HINDCAST_O3_DIR, "*.cam.h3.O3.nc")))
print(f"[INFO] 共找到 {len(hindcast_files)} 个 Hindcast 成员文件。")

results = []
for f in hindcast_files:
    member_id = os.path.basename(f).split(".")[5]  # 解析 '001', '015' 等
    h_o3 = get_polar_mean_jan_to_may(f)
    
    # 对齐时间长度计算 RMSE (积分平方差也可以，RMSE单位与DU一致更直观)
    min_len = min(len(ref_o3), len(h_o3))
    rmse = float(np.sqrt(np.mean((h_o3.values[:min_len] - ref_o3.values[:min_len])**2)))
    
    results.append({
        "Member": member_id,
        "RMSE_DU": rmse,
        "File": f
    })
    print(f"  -> Member {member_id} | RMSE: {rmse:.2f} DU")

df = pd.DataFrame(results)
df_sorted = df.sort_values(by="RMSE_DU", ascending=True).reset_index(drop=True)
df_sorted.index += 1  # 排名 1 到 N

# 保存 TXT
with open(TXT_OUT, "w") as f:
    f.write("Hindcast 0008-01 O3 Evaluation vs BWCN 0008\n")
    f.write("Metric: RMSE of 60-90N, 30-70hPa Partial Column O3 (DU)\n")
    f.write("Period: Jan 01 to May 30\n")
    f.write("="*60 + "\n")
    f.write(f"{'Rank':<5} | {'Member':<10} | {'RMSE (DU)':<15}\n")
    f.write("-" * 36 + "\n")
    for rank, row in df_sorted.iterrows():
        f.write(f"{rank:<5} | {row['Member']:<10} | {row['RMSE_DU']:<15.4f}\n")

# 保存 CSV 供下个脚本读取
df_sorted.to_csv(CSV_OUT, index_label="Rank")
print(f"\n[SUCCESS] 结果已保存至:\n  TXT: {TXT_OUT}\n  CSV: {CSV_OUT}")

In [ ]:
import os
import glob
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

# ============================================================
# 1. 路径、绘图配置与对比参数
# ============================================================
HINDCAST_BASE = "/mnt/soclim0/public_data/weiji/Hindcast/0008-01"
CSV_IN = "/mnt/soclim0/public_data/weiji/Hindcast/0008-01/O3_eval/Hindcast_0008-01_O3_RMSE_Ranking.csv"
PLOT_OUT = "/mnt/soclim0/public_data/weiji/Hindcast/0008-01/O3_eval/plots"
os.makedirs(PLOT_OUT, exist_ok=True)

# 💡 新增参数：选择对比最好和最差的 N 个成员
COMPARE_N = 5  

VAR_INFO = {
    "U": {"dir": "U", "nc_var": "U"},
    "T": {"dir": "T", "nc_var": "T"},
    "O3": {"dir": "O3", "nc_var": "O3"},
    "EPFLUX": {"dir": "EPflux_daily", "nc_var": "EPFLUX"} 
}

PLOT_CFG = {
    "U": {"cmap": "RdBu_r", "label": "U (m/s)", "title": "U Zonal Wind (60°N)", "sym": True},
    "T": {"cmap": "Spectral_r", "label": "T (K)", "title": "Temperature (60-90°N)", "sym": False},
    "O3": {"cmap": "viridis", "label": "O$_3$ (ppmv)", "title": "Ozone (60-90°N)", "sym": False},
    "EPFLUX": {"cmap": "PuOr_r", "label": "Fz", "title": "EP Flux Fz", "sym": True},
}

mpl.rc("xtick", direction="out", labelsize=9)
mpl.rc("ytick", direction="out", labelsize=9)

# 生成我们要插值的目标标准气压层 (1~100 hPa，对数均匀分布 31 层)
TARGET_PLEV = np.logspace(np.log10(1.0), np.log10(100.0), 31)

# ============================================================
# 2. 严谨的插值与数据提取函数
# ============================================================
def extract_profile(member_id, var_key):
    """提取 1-100hPa，Jan1-May30 的 plev x time 剖面 (包含严谨的垂直插值)"""
    mid_str = str(member_id).zfill(3)
    vinfo = VAR_INFO[var_key]
    folder = os.path.join(HINDCAST_BASE, vinfo["dir"])
    
    files = glob.glob(os.path.join(folder, f"*.{mid_str}.*.nc"))
    if not files:
        return None
        
    ds = xr.open_dataset(files[0])
    nc_var = vinfo["nc_var"] if vinfo["nc_var"] in ds else list(ds.data_vars)[0]
    
    # 截取 1月1日 - 5月30日
    mask_time = (ds.time.dt.month >= 1) & (ds.time.dt.month <= 5)
    mask_time = mask_time & ~((ds.time.dt.month == 5) & (ds.time.dt.day == 31))
    ds_sub = ds.isel(time=mask_time)
    
    # ======== 情景 A: EPFLUX (已预处理为 plev x time) ========
    if var_key == "EPFLUX":
        da = ds_sub[nc_var]
        if "lev" in da.dims: da = da.rename({"lev": "plev"})
        if "plev" in da.dims:
            # Pa 转 hPa
            if da["plev"].max().values > 2000:
                da = da.assign_coords(plev=da["plev"] / 100.0)
            da = da.where((da.plev >= 1.0) & (da.plev <= 100.0), drop=True)
        if "plev" in da.dims and "time" in da.dims:
            da = da.transpose("plev", "time")
        res = da.compute()
        ds.close()
        return res

    # ======== 情景 B: U, T, O3 (需要在求平均前，先插值到标准气压层) ========
    # 1. 空间截取 (极大地减少需要插值的数据量)
    if var_key == "U" and "lat" in ds_sub.dims:
        ds_sub = ds_sub.sel(lat=60.0, method="nearest")
    elif var_key in ["T", "O3"] and "lat" in ds_sub.dims:
        ds_sub = ds_sub.sel(lat=slice(60.0, 90.0))

    # 2. 计算每个格点真实的 3D 气压 (hPa)
    P0 = ds_sub['P0'].values if 'P0' in ds_sub else 100000.0
    p_3d = (ds_sub['hyam'] * P0 + ds_sub['hybm'] * ds_sub['PS']) / 100.0
    da_var = ds_sub[nc_var]

    # 3. 核心对数气压插值函数 (1D 沿 lev 维度)
    def _interp1d(v, p):
        m = np.isfinite(v) & np.isfinite(p) & (p > 0)
        if m.sum() < 2: 
            return np.full(len(TARGET_PLEV), np.nan)
        # np.interp 要求 x 坐标严格递增
        idx = np.argsort(p[m])
        return np.interp(np.log(TARGET_PLEV), np.log(p[m][idx]), v[m][idx], left=np.nan, right=np.nan)

    # 4. 执行插值
    da_interp = xr.apply_ufunc(
        _interp1d, da_var, p_3d,
        input_core_dims=[['lev'], ['lev']],
        output_core_dims=[['plev']],
        vectorize=True,
        output_dtypes=[float],
        dask="allowed"
    )
    da_interp = da_interp.assign_coords(plev=TARGET_PLEV)

    # 5. 插值完成后，再执行水平平均
    if "lon" in da_interp.dims:
        da_interp = da_interp.mean(dim="lon")
        
    if var_key in ["T", "O3"] and "lat" in da_interp.dims:
        w = np.cos(np.deg2rad(da_interp["lat"]))
        da_interp = da_interp.weighted(w).mean(dim="lat")
            
    if var_key == "O3":
        da_interp = da_interp * 1e6

    da_interp = da_interp.transpose("plev", "time")
    
    res = da_interp.compute()
    ds.close()
    return res

# ============================================================
# 3. 主绘图流程
# ============================================================
df = pd.read_csv(CSV_IN, dtype={"Member": str})

# 防御性编程：防止要求的 COMPARE_N 大于实际文件数
top_n = min(COMPARE_N, len(df))

best_members = df.head(top_n)["Member"].tolist()
worst_members = df.tail(top_n)["Member"].tolist()

print(f"[INFO] 最佳的 {top_n} 个成员: {best_members}")
print(f"[INFO] 最差的 {top_n} 个成员: {worst_members}")

for var_key in ["U", "T", "O3", "EPFLUX"]:
    print(f"\n[INFO] 正在插值并处理变量: {var_key} (可能会需要几秒钟...)")
    
    best_das = [extract_profile(mid, var_key) for mid in best_members]
    best_das = [d for d in best_das if d is not None]
    
    worst_das = [extract_profile(mid, var_key) for mid in worst_members]
    worst_das = [d for d in worst_das if d is not None]
        
    if not best_das or not worst_das:
        print(f"  -> 无法集齐 {var_key} 的有效数据，跳过绘图。")
        continue
        
    da_best = xr.concat(best_das, dim="member").mean(dim="member")
    da_worst = xr.concat(worst_das, dim="member").mean(dim="member")
    
    x_days = np.arange(da_best.sizes["time"])
    plev = da_best["plev"].values
    
    z_all = np.concatenate([da_best.values.flatten(), da_worst.values.flatten()])
    z_all = z_all[np.isfinite(z_all)]
    cfg = PLOT_CFG[var_key]
    
    if cfg["sym"]:
        vmax = np.nanpercentile(np.abs(z_all), 98)
        levels = np.linspace(-vmax, vmax, 21)
    else:
        vmin, vmax = np.nanpercentile(z_all, 2), np.nanpercentile(z_all, 98)
        levels = np.linspace(vmin, vmax, 21)
        
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharey=True, constrained_layout=True)
    
    # 💡 动态生成带参数的标题
    titles = [f"Best {top_n} (Low RMSE)", f"Worst {top_n} (High RMSE)"]
    
    for ax, da, title in zip(axes, [da_best, da_worst], titles):
        z = da.values
        if var_key == "EPFLUX": z = -z 
        
        cf = ax.contourf(x_days, plev, z, levels=levels, cmap=cfg["cmap"], extend="both")
        cs = ax.contour(x_days, plev, z, levels=10, colors="k", linewidths=0.5, alpha=0.6)
        ax.clabel(cs, inline=True, fontsize=6, fmt="%.1f")
        if cfg["sym"]:
            ax.contour(x_days, plev, z, levels=[0], colors="k", linewidths=1.2)
            
        ax.set_yscale("log")
        ax.invert_yaxis()
        ax.set_ylim(100, 1)
        ax.set_yticks([100, 50, 30, 10, 5, 1])
        ax.yaxis.set_major_formatter(ScalarFormatter())
        
        ax.set_xticks([0, 31, 60, 91, 121])
        ax.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May"])
        
        ax.set_title(title)
        ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)

    axes[0].set_ylabel("Pressure (hPa)")
    cbar = fig.colorbar(cf, ax=axes, pad=0.02, shrink=0.85)
    cbar.set_label(cfg["label"])
    
    fig.suptitle(cfg["title"] + " (Jan-May)", fontsize=14)
    out_png = os.path.join(PLOT_OUT, f"{var_key}_Best_vs_Worst.png")
    plt.savefig(out_png, dpi=300)
    plt.show()

print("\n[SUCCESS] 所有绘图完毕！物理场插值已完美修复。图表保存在:", PLOT_OUT)

In [ ]:
import os
import glob
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

# ============================================================
# 1. 路径、绘图配置与对比参数
# ============================================================
HINDCAST_BASE = "/mnt/soclim0/public_data/weiji/Hindcast/0008-01"
CSV_IN = "/mnt/soclim0/public_data/weiji/Hindcast/0008-01/O3_eval/Hindcast_0008-01_O3_RMSE_Ranking.csv"
PLOT_OUT = "/home/weiji/restart_exam/code/20260208HINDCAST_SUPPLY/plots"
os.makedirs(PLOT_OUT, exist_ok=True)

# 💡 核心可调参数：选择对比最好和最差的 N 个成员
COMPARE_N = 5  

# BWCN 参考数据路径
REF_FILES = {
    "U": "/mnt/soclim0/public_data/weiji/BWCN/U/BWCN.cam.h3.0008.U.nc",
    "T": "/mnt/soclim0/public_data/weiji/BWCN/T/BWCN.cam.h3.0008.T.nc",
    "O3": "/mnt/soclim0/public_data/weiji/BWCN/O3/BWCN.cam.h3.0008.O3.nc",
    "EPFLUX": "/mnt/soclim0/public_data/weiji/BWCN/EPflux_daily/EPFLUX_BWCN_Daily_24yr.nc"
}

# 💡 新增 T_min 指向 T 文件，专用于提取极区最低温度
VAR_INFO = {
    "U": {"dir": "U", "nc_var": "U"},
    "T": {"dir": "T", "nc_var": "T"},
    "T_min": {"dir": "T", "nc_var": "T"}, 
    "O3": {"dir": "O3", "nc_var": "O3"},
    "EPFLUX": {"dir": "EPflux_daily", "nc_var": "EPFLUX"} 
}

# 偏差图 (Anomaly) 使用发散型色标，强制对称 (sym=True)
PLOT_CFG_DIFF = {
    "U": {"cmap": "RdBu_r", "label": "ΔU (m/s)", "title": "U Bias (60°N)", "sym": True},
    "T": {"cmap": "RdBu_r", "label": "ΔT (K)", "title": "Temperature Bias (60-90°N)", "sym": True},
    "O3": {"cmap": "BrBG", "label": "ΔO$_3$ (ppmv)", "title": "Ozone Bias (60-90°N)", "sym": True},
    "EPFLUX": {"cmap": "PuOr_r", "label": "ΔFz", "title": "EP Flux Fz Bias", "sym": True},
}

mpl.rc("xtick", direction="out", labelsize=9)
mpl.rc("ytick", direction="out", labelsize=9)

TARGET_PLEV = np.logspace(np.log10(1.0), np.log10(100.0), 31)

# ============================================================
# 2. 严谨的插值与数据提取核心逻辑
# ============================================================
def process_single_file(file_path, var_key):
    """通用单文件处理核心函数：切片 -> 计算3D气压 -> 插值 -> 空间平均/极小值"""
    vinfo = VAR_INFO[var_key]
    
    ds = xr.open_dataset(file_path)
    
    # 获取变量名 (特别处理 EPFLUX 参考文件里的 Fz)
    if var_key == "EPFLUX":
        nc_var = "Fz" if "Fz" in ds else "EPFLUX"
    else:
        nc_var = vinfo["nc_var"] if vinfo["nc_var"] in ds else list(ds.data_vars)[0]
    
    # 截取时间逻辑 (支持 24 年合并文件中精准提取第 8 年)
    years_in_ds = np.unique(ds.time.dt.year)
    if len(years_in_ds) > 1:
        mask_time = (ds.time.dt.year == 8)
    else:
        mask_time = xr.ones_like(ds.time, dtype=bool)
        
    mask_time = mask_time & (ds.time.dt.month >= 1) & (ds.time.dt.month <= 5)
    mask_time = mask_time & ~((ds.time.dt.month == 5) & (ds.time.dt.day == 31))
    ds_sub = ds.isel(time=mask_time)
    
    # ======== 情景 A: EPFLUX ========
    if var_key == "EPFLUX":
        da = ds_sub[nc_var]
        if "lev" in da.dims:
            da = da.rename({"lev": "plev"})
        if "plev" in da.dims:
            if da["plev"].max().values > 2000:
                da = da.assign_coords(plev=da["plev"] / 100.0)
            da = da.where((da.plev >= 1.0) & (da.plev <= 100.0), drop=True)
        if "plev" in da.dims and "time" in da.dims:
            da = da.transpose("plev", "time")
        res = da.compute()
        ds.close()
        return res

    # ======== 情景 B: U, T, T_min, O3 ========
    if var_key == "U" and "lat" in ds_sub.dims:
        ds_sub = ds_sub.sel(lat=60.0, method="nearest")
    elif var_key in ["T", "T_min", "O3"] and "lat" in ds_sub.dims:
        ds_sub = ds_sub.sel(lat=slice(60.0, 90.0))

    P0 = ds_sub['P0'].values if 'P0' in ds_sub else 100000.0
    p_3d = (ds_sub['hyam'] * P0 + ds_sub['hybm'] * ds_sub['PS']) / 100.0
    da_var = ds_sub[nc_var]

    def _interp1d(v, p):
        m = np.isfinite(v) & np.isfinite(p) & (p > 0)
        if m.sum() < 2:
            return np.full(len(TARGET_PLEV), np.nan)
        idx = np.argsort(p[m])
        return np.interp(np.log(TARGET_PLEV), np.log(p[m][idx]), v[m][idx], left=np.nan, right=np.nan)

    da_interp = xr.apply_ufunc(
        _interp1d, da_var, p_3d,
        input_core_dims=[['lev'], ['lev']],
        output_core_dims=[['plev']],
        vectorize=True,
        output_dtypes=[float],
        dask="allowed"
    )
    da_interp = da_interp.assign_coords(plev=TARGET_PLEV)

    # 先 zonal mean
    if "lon" in da_interp.dims:
        da_interp = da_interp.mean(dim="lon")

    # 再做极区平均 / 极区最小值
    if var_key in ["T", "O3"] and "lat" in da_interp.dims:
        w = np.cos(np.deg2rad(da_interp["lat"]))
        da_interp = da_interp.weighted(w).mean(dim="lat")
    elif var_key == "T_min" and "lat" in da_interp.dims:
        da_interp = da_interp.min(dim="lat")
            
    if var_key == "O3":
        da_interp = da_interp * 1e6

    da_interp = da_interp.transpose("plev", "time")
    
    res = da_interp.compute()
    ds.close()
    return res

def extract_hindcast_profile(member_id, var_key):
    """提取特定 Hindcast 成员数据"""
    mid_str = str(member_id).zfill(3)
    folder = os.path.join(HINDCAST_BASE, VAR_INFO[var_key]["dir"])
    files = glob.glob(os.path.join(folder, f"*.{mid_str}.*.nc"))
    if not files: return None
    return process_single_file(files[0], var_key)

def extract_ref_profile(var_key):
    """提取 BWCN 参考场数据"""
    file_path = REF_FILES.get(var_key.replace("T_min", "T")) # T_min 也用 T 的参考文件
    if not file_path or not os.path.exists(file_path):
        print(f"[警告] 找不到 {var_key} 的参考文件: {file_path}")
        return None
    return process_single_file(file_path, var_key)

# ============================================================
# 3. 主绘图流程 (绘制偏差图 + 叠加绝对温度阈值)
# ============================================================
df = pd.read_csv(CSV_IN, dtype={"Member": str})
top_n = min(COMPARE_N, len(df))

best_members = df.head(top_n)["Member"].tolist()
worst_members = df.tail(top_n)["Member"].tolist()

print(f"[INFO] 偏差图模式 (Anomaly = Hindcast - Reference)")
print(f"[INFO] 最佳的 {top_n} 个成员: {best_members}")
print(f"[INFO] 最差的 {top_n} 个成员: {worst_members}")

for var_key in ["U", "T", "O3", "EPFLUX"]:
    print(f"\n[INFO] 正在处理变量: {var_key} 的偏差计算...")
    
    # 1. 提取参考场
    da_ref = extract_ref_profile(var_key)
    if da_ref is None:
        print(f"  -> 无法获取 {var_key} 的参考真值，跳过偏差图绘制。")
        continue

    # 2. 提取 Hindcast 集合
    best_das = [extract_hindcast_profile(mid, var_key) for mid in best_members]
    best_das = [d for d in best_das if d is not None]
    
    worst_das = [extract_hindcast_profile(mid, var_key) for mid in worst_members]
    worst_das = [d for d in worst_das if d is not None]
        
    if not best_das or not worst_das:
        print(f"  -> 无法集齐 {var_key} 的有效 Hindcast 数据，跳过绘图。")
        continue
        
    # 如果是温度，我们额外提取一次极区最低温度 (T_min)
    if var_key == "T":
        da_ref_tmin = extract_ref_profile("T_min")
        
        best_das_tmin = [extract_hindcast_profile(mid, "T_min") for mid in best_members]
        best_das_tmin = [d for d in best_das_tmin if d is not None]
        
        worst_das_tmin = [extract_hindcast_profile(mid, "T_min") for mid in worst_members]
        worst_das_tmin = [d for d in worst_das_tmin if d is not None]
        
        da_best_tmin_mean = xr.concat(best_das_tmin, dim="member").mean(dim="member")
        da_worst_tmin_mean = xr.concat(worst_das_tmin, dim="member").mean(dim="member")
        
    # 3. 计算集合平均并做差值 (Hindcast - Reference)
    da_best_mean = xr.concat(best_das, dim="member").mean(dim="member")
    da_worst_mean = xr.concat(worst_das, dim="member").mean(dim="member")
    
    # 对齐时间长度防止报错
    min_len = min(da_best_mean.sizes["time"], da_ref.sizes["time"])
    da_best_diff = da_best_mean.isel(time=slice(0, min_len)) - da_ref.isel(time=slice(0, min_len))
    da_worst_diff = da_worst_mean.isel(time=slice(0, min_len)) - da_ref.isel(time=slice(0, min_len))
    
    x_days = np.arange(min_len)
    plev = da_best_diff["plev"].values
    
    # 4. 确定颜色刻度
    z_all = np.concatenate([da_best_diff.values.flatten(), da_worst_diff.values.flatten()])
    z_all = z_all[np.isfinite(z_all)]
    cfg = PLOT_CFG_DIFF[var_key]
    
    vmax = np.nanpercentile(np.abs(z_all), 98)
    if vmax == 0 or np.isnan(vmax): vmax = 1.0
    levels = np.linspace(-vmax, vmax, 21)
        
    # 5. 开始绘制并排偏差图
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharey=True, constrained_layout=True)
    
    titles = [f"Best {top_n} Bias (Hindcast - Ref)", f"Worst {top_n} Bias (Hindcast - Ref)"]
    is_best_flags = [True, False]
    
    for ax, da_diff, title, is_best in zip(axes, [da_best_diff, da_worst_diff], titles, is_best_flags):
        z = da_diff.values
        if var_key == "EPFLUX": z = -z 
        
        cf = ax.contourf(x_days, plev, z, levels=levels, cmap=cfg["cmap"], extend="both")
        
        cs = ax.contour(x_days, plev, z, levels=10, colors="k", linewidths=0.5, alpha=0.6)
        ax.clabel(cs, inline=True, fontsize=6, fmt="%.1f")
        
        # ============================================================
        # 💡 温度专属逻辑：基于极地最低温度 T_min 绘制 197K 阈值
        # ============================================================
        if var_key == "T":
            da_tmin_mean = da_best_tmin_mean if is_best else da_worst_tmin_mean
            
            ref_z_min = da_ref_tmin.isel(time=slice(0, min_len)).values
            mean_z_min = da_tmin_mean.isel(time=slice(0, min_len)).values
            
            print(f"    [Diagnostic] {title} - 参考场最低温度: {np.nanmin(ref_z_min):.1f} K")
            print(f"    [Diagnostic] {title} - 预测场最低温度: {np.nanmin(mean_z_min):.1f} K")
            
            t_levels = [100, 197] 
            
            # 参考数据 T_min < 197K：/// 阴影 + 虚线轮廓
            ax.contourf(x_days, plev, ref_z_min, levels=t_levels, 
                        hatches=['///'], colors='none')
            ax.contour(x_days, plev, ref_z_min, levels=[197], 
                       colors='black', linestyles='dashed', linewidths=1.5, alpha=0.8)
            
            # 预测数据 T_min < 197K：\\\\ 阴影 + 实线轮廓
            ax.contourf(x_days, plev, mean_z_min, levels=t_levels, 
                        hatches=['\\\\\\\\'], colors='none')
            ax.contour(x_days, plev, mean_z_min, levels=[197], 
                       colors='black', linestyles='solid', linewidths=1.5, alpha=0.8)
            
            # 添加文本说明
            ax.text(0.02, 0.03, "Hatches: Polar Min T < 197K", 
                    transform=ax.transAxes, fontsize=8, fontweight='bold',
                    bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=2))
        # ============================================================

        # 💡❗ 新增逻辑：在 O3 图上也叠加 T Minimum 区域 ❗💡
        if var_key == "O3":
            # 1. 临时提取 T_min 数据（已移至 plotting loop 内以确保可访问）
            da_ref_tmin_for_o3 = extract_ref_profile("T_min")
            
            best_das_tmin_for_o3 = [extract_hindcast_profile(mid, "T_min") for mid in best_members]
            best_das_tmin_for_o3 = [d for d in best_das_tmin_for_o3 if d is not None]
            
            worst_das_tmin_for_o3 = [extract_hindcast_profile(mid, "T_min") for mid in worst_members]
            worst_das_tmin_for_o3 = [d for d in worst_das_tmin_for_o3 if d is not None]
            
            da_best_tmin_mean_for_o3 = xr.concat(best_das_tmin_for_o3, dim="member").mean(dim="member")
            da_worst_tmin_mean_for_o3 = xr.concat(worst_das_tmin_for_o3, dim="member").mean(dim="member")

            da_tmin_mean_for_o3 = da_best_tmin_mean_for_o3 if is_best else da_worst_tmin_mean_for_o3
            
            # 2. 提取对齐后的数据（使用新的临时变量名防止覆盖）
            ref_z_min_for_o3 = da_ref_tmin_for_o3.isel(time=slice(0, min_len)).values
            mean_z_min_for_o3 = da_tmin_mean_for_o3.isel(time=slice(0, min_len)).values
            
            t_levels = [100, 197] 
            
            # 3. 复制视觉移植逻辑：完全相同的 hatches 和 contours！
            # 参考数据 T_min < 197K：/// 阴影 + 虚线轮廓
            ax.contourf(x_days, plev, ref_z_min_for_o3, levels=t_levels, 
                        hatches=['///'], colors='none')
            ax.contour(x_days, plev, ref_z_min_for_o3, levels=[197], 
                       colors='black', linestyles='dashed', linewidths=1.5, alpha=0.8)
            
            # 预测数据 T_min < 197K：\\\\ 阴影 + 实线轮廓
            ax.contourf(x_days, plev, mean_z_min_for_o3, levels=t_levels, 
                        hatches=['\\\\\\\\'], colors='none')
            ax.contour(x_days, plev, mean_z_min_for_o3, levels=[197], 
                       colors='black', linestyles='solid', linewidths=1.5, alpha=0.8)
            
            # 4. 添加文本说明
            ax.text(0.02, 0.03, "Hatches: Polar Min T < 197K", 
                    transform=ax.transAxes, fontsize=8, fontweight='bold',
                    bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=2))
        # ============================================================

        ax.set_yscale("log")
        ax.invert_yaxis()
        ax.set_ylim(100, 1)
        ax.set_yticks([100, 50, 30, 10, 5, 1])
        ax.yaxis.set_major_formatter(ScalarFormatter())
        
        ax.set_xticks([0, 31, 60, 91, 121])
        ax.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May"])
        
        ax.set_title(title)
        ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)

    axes[0].set_ylabel("Pressure (hPa)")
    cbar = fig.colorbar(cf, ax=axes, pad=0.02, shrink=0.85)
    cbar.set_label(cfg["label"])
    
    fig.suptitle(cfg["title"] + " (Jan-May)", fontsize=14)
    out_png = os.path.join(PLOT_OUT, f"{var_key}_Bias_Best{COMPARE_N}_vs_Worst{COMPARE_N}.png")
    plt.savefig(out_png, dpi=300)
    plt.show()

print("\n[SUCCESS] 偏差图绘制完毕！图表保存在:", PLOT_OUT)

In [ ]:
import os
import glob
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# 1. 路径与参数设置
# ============================================================
HINDCAST_BASE = "/mnt/soclim0/public_data/weiji/Hindcast/0008-01"
CSV_IN = "/mnt/soclim0/public_data/weiji/Hindcast/0008-01/O3_eval/Hindcast_0008-01_O3_RMSE_Ranking.csv"
PLOT_OUT = "/home/weiji/restart_exam/code/20260208HINDCAST_SUPPLY/plots"
os.makedirs(PLOT_OUT, exist_ok=True)

# BWCN 参考文件路径 (请确保真实存在)
REF_FILES = {
    "U": "/mnt/soclim0/public_data/weiji/BWCN/U/BWCN.cam.h3.0008.U.nc",
    "T": "/mnt/soclim0/public_data/weiji/BWCN/T/BWCN.cam.h3.0008.T.nc",
    "O3": "/mnt/soclim0/public_data/weiji/BWCN/O3/BWCN.cam.h3.0008.O3.nc",
    "EPFLUX": "/mnt/soclim0/public_data/weiji/BWCN/EPflux_daily/EPFLUX_BWCN_Daily_24yr.nc"
}

COMPARE_N = 5  

# ============================================================
# 2. 核心数据提取函数
# ============================================================
def _get_time_mask(ds):
    mask = (ds.time.dt.month >= 1) & (ds.time.dt.month <= 5)
    mask = mask & ~((ds.time.dt.month == 5) & (ds.time.dt.day == 31))
    return mask

# --- (1) 提取 U: 60°N, 10 hPa ---
def get_u_series(file_path):
    ds = xr.open_dataset(file_path)
    nc_var = "U" if "U" in ds else list(ds.data_vars)[0]
    
    ds_sub = ds.isel(time=_get_time_mask(ds))
    if "lon" in ds_sub.dims: ds_sub = ds_sub.mean(dim="lon")
    if "lat" in ds_sub.dims: ds_sub = ds_sub.sel(lat=60.0, method="nearest")
        
    P0 = ds_sub['P0'].values if 'P0' in ds_sub else 100000.0
    p_real = ((ds_sub['hyam'] * P0 + ds_sub['hybm'] * ds_sub['PS']) / 100.0).transpose("time", "lev").values
    u_raw = ds_sub[nc_var].transpose("time", "lev").values
    
    n_time = u_raw.shape[0]
    u_10hpa = np.zeros(n_time)
    for t in range(n_time):
        p_t, u_t = p_real[t, :], u_raw[t, :]
        valid = np.isfinite(p_t) & np.isfinite(u_t) & (p_t > 0)
        if valid.sum() < 2:
            u_10hpa[t] = np.nan
        else:
            idx = np.argsort(p_t[valid])
            u_10hpa[t] = np.interp(np.log(10.0), np.log(p_t[valid][idx]), u_t[valid][idx])
            
    ds.close()
    return u_10hpa

# --- (2) 提取 T: 50 hPa, 纬向平均的极小值 (Zonal Mean Minimum) ---
def get_t_min_series(file_path):
    ds = xr.open_dataset(file_path)
    nc_var = "T" if "T" in ds else list(ds.data_vars)[0]
    
    ds_sub = ds.isel(time=_get_time_mask(ds)).sel(lat=slice(60.0, 90.0))
    P0 = ds_sub['P0'].values if 'P0' in ds_sub else 100000.0
    p_3d = (ds_sub['hyam'] * P0 + ds_sub['hybm'] * ds_sub['PS']) / 100.0
    
    def _interp_50(v, p):
        m = np.isfinite(v) & np.isfinite(p) & (p > 0)
        if m.sum() < 2: return np.nan
        idx = np.argsort(p[m])
        return np.interp(np.log(50.0), np.log(p[m][idx]), v[m][idx], left=np.nan, right=np.nan)

    t_50hpa = xr.apply_ufunc(
        _interp_50, ds_sub[nc_var], p_3d,
        input_core_dims=[['lev'], ['lev']],
        vectorize=True, output_dtypes=[float], dask="allowed"
    )
    
    # 【核心修改】：先求纬向平均(mean lon)，再求纬向的最小值(min lat)
    if "lon" in t_50hpa.dims:
        t_50hpa = t_50hpa.mean(dim="lon")
    t_min = t_50hpa.min(dim="lat").values
    
    ds.close()
    return t_min

# --- (3) 提取 O3: 60-90°N, 30-70 hPa Partial Column ---
def calc_parc_o3_hybrid(ds_sub, p_top_hpa, p_bot_hpa):
    g, M_air, Na = 9.80665, 28.964 / 1000.0, 6.02214e23
    factor = Na / (g * M_air * 2.687e20)
    P0, PS = ds_sub['P0'], ds_sub['PS']
    P_interface = ds_sub['hyai'] * P0 + ds_sub['hybi'] * PS
    
    p_i = P_interface.isel(ilev=slice(0, -1)).rename({'ilev': 'lev'}).assign_coords(lev=ds_sub['lev'])
    p_ip1 = P_interface.isel(ilev=slice(1, None)).rename({'ilev': 'lev'}).assign_coords(lev=ds_sub['lev'])
    
    p_layer_top = xr.where(p_i < p_ip1, p_i, p_ip1)
    p_layer_bot = xr.where(p_i > p_ip1, p_i, p_ip1)
    
    pT, pB = p_top_hpa * 100.0, p_bot_hpa * 100.0
    upper = xr.where(p_layer_top > pT, p_layer_top, pT)
    lower = xr.where(p_layer_bot < pB, p_layer_bot, pB)
    
    overlap = xr.where(lower > upper, lower - upper, 0.0)
    return (ds_sub['O3'] * overlap * factor).sum(dim='lev')

def get_o3_series(file_path):
    ds = xr.open_dataset(file_path)
    ds_sub = ds.isel(time=_get_time_mask(ds))
    
    o3_col = calc_parc_o3_hybrid(ds_sub, 30.0, 70.0)
    o3_nh = o3_col.sel(lat=slice(60.0, 90.0))
    weights = np.cos(np.deg2rad(o3_nh.lat))
    o3_mean = o3_nh.weighted(weights).mean(dim=["lat", "lon"]).values
    
    ds.close()
    return o3_mean

# --- (4) 提取 EP FLUX: 100 hPa ---
def get_epflux_100hpa(file_path):
    ds = xr.open_dataset(file_path)
    nc_var = "Fz" if "Fz" in ds else "EPFLUX"
    
    # 支持多年文件的自动截取
    years_in_ds = np.unique(ds.time.dt.year)
    if len(years_in_ds) > 1:
        mask_time = (ds.time.dt.year == 8)
    else:
        mask_time = xr.ones_like(ds.time, dtype=bool)
        
    mask_time = mask_time & (ds.time.dt.month >= 1) & (ds.time.dt.month <= 5)
    mask_time = mask_time & ~((ds.time.dt.month == 5) & (ds.time.dt.day == 31))
    ds_sub = ds.isel(time=mask_time)
    
    da = ds_sub[nc_var]
    if "lev" in da.dims: da = da.rename({"lev": "plev"})
    
    # 取 100 hPa (如果是 Pa 则取 10000.0)
    if "plev" in da.dims:
        if da["plev"].max().values > 2000:
            val_100hpa = da.sel(plev=10000.0, method="nearest")
        else:
            val_100hpa = da.sel(plev=100.0, method="nearest")
    
    # 🚨【核心修复】：这里加上负号
    res = -val_100hpa.values  
    ds.close()
    return res

# ============================================================
# 3. 统一绘图函数
# ============================================================
def plot_variable_series(data_dict, ref_series, best_members, worst_members, 
                         title, ylabel, save_name, hlines=None):
    all_series = list(data_dict.values())
    if not all_series: return
    
    ens_mean = np.nanmean(all_series, axis=0)
    ens_std = np.nanstd(all_series, axis=0)
    x_days = np.arange(len(ens_mean))

    best_series = [data_dict[m] for m in best_members if m in data_dict]
    best_mean = np.nanmean(best_series, axis=0) if best_series else np.full_like(ens_mean, np.nan)

    worst_series = [data_dict[m] for m in worst_members if m in data_dict]
    worst_mean = np.nanmean(worst_series, axis=0) if worst_series else np.full_like(ens_mean, np.nan)

    fig, ax = plt.subplots(figsize=(10, 5.5), constrained_layout=True)

    # 1. 单独的成员线
    cmap = plt.cm.get_cmap("tab20", len(data_dict))
    for i, (mid, series) in enumerate(data_dict.items()):
        ax.plot(x_days, series, color=cmap(i), lw=0.6, alpha=0.35, zorder=2)

    # 2. 集合平均 ±1σ
    ax.fill_between(x_days, ens_mean - ens_std, ens_mean + ens_std, 
                    color="limegreen", alpha=0.2, zorder=3, label="Ens Mean ±1σ")

    # 3. 平均线
    ax.plot(x_days, ens_mean, color="limegreen", lw=2.5, label="All Ens Mean", zorder=5)
    ax.plot(x_days, best_mean, color="dodgerblue", lw=2.5, label=f"Best {COMPARE_N} Mean", zorder=6)
    ax.plot(x_days, worst_mean, color="crimson", lw=2.5, label=f"Worst {COMPARE_N} Mean", zorder=6)

    # 4. 参考线
    if ref_series is not None:
        min_len = min(len(x_days), len(ref_series))
        ax.plot(x_days[:min_len], ref_series[:min_len], color="black", lw=3.0, label="BWCN Reference", zorder=10)

    # 5. 自定义水平参考线 (如 195K, 197K, 0)
    if hlines:
        for hl in hlines:
            ax.axhline(hl['val'], color=hl['color'], lw=1.2, ls="--", label=hl['label'], zorder=4)

    ax.set_xlim(0, x_days[-1])
    ax.set_xticks([0, 31, 60, 91, 121])
    ax.set_xticklabels(["Jan 1", "Feb 1", "Mar 1", "Apr 1", "May 1"])
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=13)
    ax.grid(True, linestyle=":", alpha=0.6)
    ax.legend(loc="best", frameon=True, edgecolor="none", facecolor="white", framealpha=0.8, fontsize=9)

    out_path = os.path.join(PLOT_OUT, save_name)
    plt.savefig(out_path, dpi=300)
    plt.show()  # 💡 强制显示图像
    print(f"[SUCCESS] 已保存: {out_path}")

# ============================================================
# 4. 主运行流程
# ============================================================
df = pd.read_csv(CSV_IN, dtype={"Member": str})
top_n = min(COMPARE_N, len(df))

best_members = set(df.head(top_n)["Member"].tolist())
worst_members = set(df.tail(top_n)["Member"].tolist())

PROCESS_TASKS = [
    {
        "var": "U", "dir": "U", "func": get_u_series, 
        "hlines": [{"val": 0, "color": "gray", "label": "U = 0 (SSW)"}], 
        "ylabel": "U Zonal Wind (m/s)", 
        "title": f"Hindcast 0008-01: Zonal Wind at 60°N, 10 hPa", 
        "save": f"U_60N_10hPa_Line_{COMPARE_N}.png"
    },
    {
        "var": "T", "dir": "T", "func": get_t_min_series, 
        "hlines": [
                   {"val": 197, "color": "purple", "label": "Cl Act (197K)"}], 
        "ylabel": "Minimum Temperature (K)", 
        "title": f"Hindcast 0008-01: Zonal Min Temp (60-90°N, 50 hPa)", 
        "save": f"Tmin_60-90N_50hPa_Line_{COMPARE_N}.png"
    },
    {
        "var": "O3", "dir": "O3", "func": get_o3_series, 
        "hlines": [], 
        "ylabel": "Partial Column O3 (DU)", 
        "title": f"Hindcast 0008-01: Partial Column O3 (60-90°N, 30-70 hPa)", 
        "save": f"O3_Col_60-90N_Line_{COMPARE_N}.png"
    },
    {
        "var": "EPFLUX", "dir": "EPflux_daily", "func": get_epflux_100hpa, 
        "hlines": [{"val": 0, "color": "gray", "label": "Fz = 0"}], 
        "ylabel": "EP Flux Fz (native)", 
        "title": f"Hindcast 0008-01: EP Flux Fz at 100 hPa (40-80°N)", 
        "save": f"EPFLUX_100hPa_Line_{COMPARE_N}.png"
    }
]

for task in PROCESS_TASKS:
    var_key = task["var"]
    print(f"\n[INFO] 正在处理变量: {var_key} ...")
    
    # 提取 BWCN
    ref_file = REF_FILES.get(var_key)
    try:
        ref_series = task["func"](ref_file)
    except Exception as e:
        print(f"  [警告] 提取参考 {var_key} 失败: {e}")
        ref_series = None

    # 提取 Hindcast
    hind_dir = os.path.join(HINDCAST_BASE, task["dir"])
    files = sorted(glob.glob(os.path.join(hind_dir, "*.nc")))
    
    data_dict = {}
    for f in files:
        mid = os.path.basename(f).split(".")[5].zfill(3)
        try:
            data_dict[mid] = task["func"](f)
        except Exception as e:
            print(f"  [警告] 读取 {os.path.basename(f)} 失败: {e}")

    # 绘图并展示
    plot_variable_series(
        data_dict, ref_series, best_members, worst_members,
        title=task["title"], ylabel=task["ylabel"], 
        save_name=task["save"], hlines=task["hlines"]
    )

In [ ]:
import os
import glob
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress

# ============================================================
# 1. 路径设置
# ============================================================
HINDCAST_BASE = "/mnt/soclim0/public_data/weiji/Hindcast/0008-01"
CSV_IN = "/mnt/soclim0/public_data/weiji/Hindcast/0008-01/O3_eval/Hindcast_0008-01_O3_RMSE_Ranking.csv"
PLOT_OUT = "/home/weiji/restart_exam/code/20260208HINDCAST_SUPPLY/plots"
os.makedirs(PLOT_OUT, exist_ok=True)

# ============================================================
# 2. 提取 100hPa Fz 平均值 (Jan 20 - Feb 10)
# ============================================================
def get_fz_mean_jan20_feb10(member_id):
    """提取单文件 1月20日到2月10日的 100 hPa EP Flux Fz 平均值"""
    mid_str = str(member_id).zfill(3)
    folder = os.path.join(HINDCAST_BASE, "EPflux_daily")
    files = glob.glob(os.path.join(folder, f"*.{mid_str}.*.nc"))
    
    if not files:
        return np.nan
        
    ds = xr.open_dataset(files[0])
    nc_var = "Fz" if "Fz" in ds else ("EPFLUX" if "EPFLUX" in ds else list(ds.data_vars)[0])
    
    mask_jan = (ds.time.dt.month == 1) & (ds.time.dt.day >= 20)
    mask_feb = (ds.time.dt.month == 2) & (ds.time.dt.day <= 10)
    mask_time = mask_jan | mask_feb
    
    ds_sub = ds.isel(time=mask_time)
    da = ds_sub[nc_var]
    
    if "lev" in da.dims: 
        da = da.rename({"lev": "plev"})
        
    if "plev" in da.dims:
        if da["plev"].max().values > 2000:
            val_100hpa = da.sel(plev=10000.0, method="nearest")
        else:
            val_100hpa = da.sel(plev=100.0, method="nearest")
    else:
        val_100hpa = da
        
    # 🚨【核心修复】：加上负号，让向上的波活动变为正值
    fz_mean = -val_100hpa.mean(dim="time").values
    ds.close()
    
    return float(fz_mean)

# ============================================================
# 3. 数据配对处理
# ============================================================
print("[INFO] 正在读取 RMSE 排名并提取 EP Flux 数据...")
df = pd.read_csv(CSV_IN, dtype={"Member": str})

x_fz = []
y_rmse = []
valid_members = []

for idx, row in df.iterrows():
    mid = row["Member"]
    rmse = row["RMSE_DU"]
    fz_mean = get_fz_mean_jan20_feb10(mid)
    
    if np.isfinite(fz_mean):
        x_fz.append(fz_mean)
        y_rmse.append(rmse)
        valid_members.append(mid)
    else:
        print(f"  [警告] 成员 {mid} 获取 Fz 数据失败。")

x_fz = np.array(x_fz)
y_rmse = np.array(y_rmse)

# ============================================================
# 4. 绘制散点图与回归线
# ============================================================
fig, ax = plt.subplots(figsize=(7.5, 5.5), constrained_layout=True)

ax.scatter(x_fz, y_rmse, s=60, c="dodgerblue", alpha=0.8, edgecolor="k", linewidth=0.5, zorder=3)

if len(x_fz) > 1:
    slope, intercept, r_value, p_value, std_err = linregress(x_fz, y_rmse)
    x_fit = np.linspace(x_fz.min() - 0.5e5, x_fz.max() + 0.5e5, 100)
    y_fit = slope * x_fit + intercept
    
    ax.plot(x_fit, y_fit, color="crimson", lw=2, ls="--", zorder=2, label="Linear Trend")
    
    text_str = f"R = {r_value:.3f}\n$R^2$ = {r_value**2:.3f}\np-value = {p_value:.3e}"
    props = dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray')
    ax.text(0.05, 0.95, text_str, transform=ax.transAxes, fontsize=11,
            verticalalignment='top', bbox=props)

# 注意这里 X 轴标签我也加上了翻转的说明
ax.set_xlabel("Mean 100 hPa EP Flux Fz (Jan 20 - Feb 10) [-1 * native units]", fontsize=11)
ax.set_ylabel("O3 Partial Column RMSE (DU)", fontsize=11)
ax.set_title("Ozone RMSE vs. Winter Wave Activity (-Fz)", fontsize=13)

x_margin = (x_fz.max() - x_fz.min()) * 0.1
y_margin = (y_rmse.max() - y_rmse.min()) * 0.1
ax.set_xlim(x_fz.min() - x_margin, x_fz.max() + x_margin)
ax.set_ylim(y_rmse.min() - y_margin, y_rmse.max() + y_margin)

ax.grid(True, linestyle="--", alpha=0.6, zorder=1)
if len(x_fz) > 1:
    ax.legend(loc="upper right", frameon=False)

out_path = os.path.join(PLOT_OUT, "Scatter_RMSE_vs_Fz_Jan20-Feb10.png")
plt.savefig(out_path, dpi=300)
plt.show()

print(f"\n[SUCCESS] 散点图绘制完毕！图表保存在: {out_path}")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import glob
from calendar import month_abbr

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

# ============================================================
# 0. 全局配置
# ============================================================
CASE_NAME = "0008-02_v2"
TARGET_YEAR = 8               # BWCN 多年文件中抽取的物理年
START_MONTH = 2               # 0008-02_v2 -> 默认从 2 月 1 日开始
END_MONTH = 5                 # 到 5 月 30 日
DROP_MAY_31 = True
COMPARE_N = 5
SHOW_FIG = True               # 如果批量跑图想更安静，改成 False

# ----------------------------
# 路径
# ----------------------------
HINDCAST_BASE = f"/mnt/soclim0/public_data/weiji/Hindcast/{CASE_NAME}"

OUT_ROOT = f"/home/weiji/restart_exam/code/20260208HINDCAST_SUPPLY/{CASE_NAME}_analysis"
RANK_OUT_DIR = os.path.join(OUT_ROOT, "ranking")
PLOT_OUT_DIR = os.path.join(OUT_ROOT, "plots")

os.makedirs(RANK_OUT_DIR, exist_ok=True)
os.makedirs(PLOT_OUT_DIR, exist_ok=True)

TXT_OUT = os.path.join(RANK_OUT_DIR, f"{CASE_NAME}_O3_RMSE_Ranking.txt")
CSV_OUT = os.path.join(RANK_OUT_DIR, f"{CASE_NAME}_O3_RMSE_Ranking.csv")

# ----------------------------
# BWCN 参考数据
# ----------------------------
REF_FILES = {
    "U": "/mnt/soclim0/public_data/weiji/BWCN/U/BWCN.cam.h3.0008.U.nc",
    "T": "/mnt/soclim0/public_data/weiji/BWCN/T/BWCN.cam.h3.0008.T.nc",
    "O3": "/mnt/soclim0/public_data/weiji/BWCN/O3/BWCN.cam.h3.0008.O3.nc",
    "EPFLUX": "/mnt/soclim0/public_data/weiji/BWCN/EPflux_daily/EPFLUX_BWCN_Daily_24yr.nc",
}

# ----------------------------
# 变量信息
# ----------------------------
VAR_INFO = {
    "U":      {"dir": "U",            "nc_var": "U"},
    "T":      {"dir": "T",            "nc_var": "T"},
    "T_min":  {"dir": "T",            "nc_var": "T"},    # 专门给 Tmin 用
    "O3":     {"dir": "O3",           "nc_var": "O3"},
    "EPFLUX": {"dir": "EPflux_daily", "nc_var": "EPFLUX"},
}

# ----------------------------
# 基本参数
# ----------------------------
LAT_BOUNDS_O3 = (60.0, 90.0)
O3_P_TOP_HPA = 30.0
O3_P_BOT_HPA = 70.0

U_LAT = 60.0
U_PLEV_HPA = 10.0

TMIN_LAT_BOUNDS = (60.0, 90.0)
TMIN_PLEV_HPA = 50.0
TMIN_THRESH_K = 197.0

EP_PLEV_HPA = 100.0

# 目标标准气压层（用于 U / T / O3 剖面插值）
TARGET_PLEV = np.logspace(np.log10(1.0), np.log10(100.0), 31)

# ----------------------------
# 绘图样式
# ----------------------------
mpl.rc("xtick", direction="out", labelsize=9)
mpl.rc("ytick", direction="out", labelsize=9)

PLOT_CFG_ABS = {
    "U": {
        "cmap": "RdBu_r",
        "label": "U (m/s)",
        "title": "U Zonal Wind (60°N)",
        "sym": True,
    },
    "T": {
        "cmap": "Spectral_r",
        "label": "T (K)",
        "title": "Temperature (60–90°N)",
        "sym": False,
    },
    "O3": {
        "cmap": "viridis",
        "label": "O$_3$ (ppmv)",
        "title": "Ozone (60–90°N)",
        "sym": False,
    },
    "EPFLUX": {
        "cmap": "PuOr_r",
        "label": "Fz",
        "title": "EP Flux Fz",
        "sym": True,
    },
}

PLOT_CFG_DIFF = {
    "U": {
        "cmap": "RdBu_r",
        "label": "ΔU (m/s)",
        "title": "U Bias (60°N)",
        "sym": True,
    },
    "T": {
        "cmap": "RdBu_r",
        "label": "ΔT (K)",
        "title": "Temperature Bias (60–90°N)",
        "sym": True,
    },
    "O3": {
        "cmap": "BrBG",
        "label": "ΔO$_3$ (ppmv)",
        "title": "Ozone Bias (60–90°N)",
        "sym": True,
    },
    "EPFLUX": {
        "cmap": "PuOr_r",
        "label": "ΔFz",
        "title": "EP Flux Fz Bias",
        "sym": True,
    },
}

# ============================================================
# 1. 通用工具函数
# ============================================================
def maybe_show_or_close():
    if SHOW_FIG:
        plt.show()
    else:
        plt.close()


def parse_member_id(file_path):
    """从文件名里提取三位成员号，如 001, 015 ..."""
    base = os.path.basename(file_path)
    tokens = base.split(".")
    for tok in reversed(tokens):
        if tok.isdigit() and len(tok) == 3:
            return tok
    raise ValueError(f"无法从文件名解析成员号: {base}")


def build_time_mask(ds, start_month=1, end_month=5, drop_may_31=True, target_year=None):
    """统一时间掩码。支持单年文件，也支持多年文件（如 BWCN EPFLUX 24yr）"""
    mask = (ds.time.dt.month >= start_month) & (ds.time.dt.month <= end_month)

    if drop_may_31:
        mask = mask & ~((ds.time.dt.month == 5) & (ds.time.dt.day == 31))

    years_in_ds = np.unique(ds.time.dt.year.values)
    if target_year is not None and len(years_in_ds) > 1:
        mask = mask & (ds.time.dt.year == target_year)

    return mask


def get_time_ticks(time_da):
    """根据真实时间坐标自动生成 x 轴月起点刻度"""
    months = np.asarray(time_da.dt.month.values)
    days = np.asarray(time_da.dt.day.values)

    tick_pos, tick_lab = [], []
    seen = set()

    for i, (m, d) in enumerate(zip(months, days)):
        if d == 1 and int(m) not in seen:
            tick_pos.append(i)
            tick_lab.append(month_abbr[int(m)])
            seen.add(int(m))

    if len(tick_pos) == 0:
        tick_pos = [0]
        tick_lab = ["Start"]

    return tick_pos, tick_lab


def concat_members(da_list, member_ids=None):
    """先按 time 做 inner 对齐，再 concat 成 member 维"""
    if len(da_list) == 0:
        return None

    aligned = xr.align(*da_list, join="inner")
    if member_ids is None:
        member_ids = np.arange(len(aligned))
    return xr.concat(aligned, dim=pd.Index(member_ids, name="member"))


def open_dataset_safely(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"文件不存在: {file_path}")
    return xr.open_dataset(file_path)


# ============================================================
# 2. O3 partial column 计算
# ============================================================
def calc_parc_o3_hybrid(ds_sub, p_top_hpa, p_bot_hpa):
    """
    在 hybrid sigma-pressure 层上精确积分 O3 部分柱（DU）
    """
    g = 9.80665
    M_air = 28.964 / 1000.0
    Na = 6.02214e23
    factor = Na / (g * M_air * 2.687e20)

    P0 = ds_sub["P0"]
    PS = ds_sub["PS"]
    P_interface = ds_sub["hyai"] * P0 + ds_sub["hybi"] * PS

    p_i = P_interface.isel(ilev=slice(0, -1)).rename({"ilev": "lev"}).assign_coords(lev=ds_sub["lev"])
    p_ip1 = P_interface.isel(ilev=slice(1, None)).rename({"ilev": "lev"}).assign_coords(lev=ds_sub["lev"])

    p_layer_top = xr.where(p_i < p_ip1, p_i, p_ip1)
    p_layer_bot = xr.where(p_i > p_ip1, p_i, p_ip1)

    pT = p_top_hpa * 100.0
    pB = p_bot_hpa * 100.0

    upper = xr.where(p_layer_top > pT, p_layer_top, pT)
    lower = xr.where(p_layer_bot < pB, p_layer_bot, pB)

    overlap = xr.where(lower > upper, lower - upper, 0.0)

    return (ds_sub["O3"] * overlap * factor).sum(dim="lev")


def get_o3_partial_column_series(file_path, start_month=1, end_month=5, drop_may_31=True, target_year=None):
    """
    返回 60–90N, 30–70 hPa 部分臭氧柱极区平均时间序列（DU）
    """
    with open_dataset_safely(file_path) as ds:
        mask = build_time_mask(
            ds,
            start_month=start_month,
            end_month=end_month,
            drop_may_31=drop_may_31,
            target_year=target_year
        )
        ds_sub = ds.isel(time=mask)
        if ds_sub.sizes.get("time", 0) == 0:
            raise ValueError(f"{file_path} 在指定时间范围内没有数据。")

        o3_col = calc_parc_o3_hybrid(ds_sub, O3_P_TOP_HPA, O3_P_BOT_HPA)
        o3_nh = o3_col.sel(lat=slice(LAT_BOUNDS_O3[0], LAT_BOUNDS_O3[1]))
        weights = np.cos(np.deg2rad(o3_nh.lat))
        o3_mean = o3_nh.weighted(weights).mean(dim=["lat", "lon"])

        return o3_mean.load()


# ============================================================
# 3. 垂直插值与剖面处理
# ============================================================
def interpolate_hybrid_to_plev(da_var, ds_sub, target_plev_hpa):
    """
    将 hybrid 层变量插值到标准气压层 (hPa)
    返回维度里包含新的 plev 维
    """
    P0 = ds_sub["P0"] if "P0" in ds_sub else 100000.0
    p_3d = (ds_sub["hyam"] * P0 + ds_sub["hybm"] * ds_sub["PS"]) / 100.0  # hPa

    def _interp1d(v, p):
        m = np.isfinite(v) & np.isfinite(p) & (p > 0)
        if m.sum() < 2:
            return np.full(len(target_plev_hpa), np.nan)

        idx = np.argsort(p[m])
        p_sorted = p[m][idx]
        v_sorted = v[m][idx]

        return np.interp(
            np.log(target_plev_hpa),
            np.log(p_sorted),
            v_sorted,
            left=np.nan,
            right=np.nan
        )

    da_interp = xr.apply_ufunc(
        _interp1d,
        da_var,
        p_3d,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True,
        output_dtypes=[np.float64],
        dask="allowed",
    )

    da_interp = da_interp.assign_coords(plev=target_plev_hpa)
    return da_interp


def process_single_profile_file(file_path, var_key, start_month=1, end_month=5, drop_may_31=True, target_year=None):
    """
    通用剖面处理函数：
      - U: 60N，先插值，再 lon mean
      - T: 60–90N，先插值，再 lon mean，再 cos(lat) 加权平均
      - T_min: 60–90N，先插值，再 zonal mean(lon)，再 min(lat)
      - O3: 60–90N，先插值，再 lon mean，再 cos(lat) 加权平均，单位转 ppmv
      - EPFLUX: 已经在 plev-time 上，直接截取
    返回 plev x time
    """
    with open_dataset_safely(file_path) as ds:
        mask = build_time_mask(
            ds,
            start_month=start_month,
            end_month=end_month,
            drop_may_31=drop_may_31,
            target_year=target_year
        )
        ds_sub = ds.isel(time=mask)
        if ds_sub.sizes.get("time", 0) == 0:
            raise ValueError(f"{file_path} 在指定时间范围内没有数据。")

        # ---------------- EPFLUX ----------------
        if var_key == "EPFLUX":
            nc_var = "Fz" if "Fz" in ds_sub else ("EPFLUX" if "EPFLUX" in ds_sub else list(ds_sub.data_vars)[0])
            da = ds_sub[nc_var]

            if "lev" in da.dims:
                da = da.rename({"lev": "plev"})

            if "plev" in da.dims:
                if da["plev"].max().values > 2000:
                    da = da.assign_coords(plev=da["plev"] / 100.0)
                da = da.where((da.plev >= 1.0) & (da.plev <= 100.0), drop=True)

            if "plev" in da.dims and "time" in da.dims:
                da = da.transpose("plev", "time")

            return da.load()

        # ---------------- U / T / T_min / O3 ----------------
        nc_var = VAR_INFO[var_key]["nc_var"]
        if nc_var not in ds_sub:
            nc_var = list(ds_sub.data_vars)[0]

        if var_key == "U" and "lat" in ds_sub.dims:
            ds_sub = ds_sub.sel(lat=U_LAT, method="nearest")

        elif var_key in ["T", "T_min", "O3"] and "lat" in ds_sub.dims:
            ds_sub = ds_sub.sel(lat=slice(TMIN_LAT_BOUNDS[0], TMIN_LAT_BOUNDS[1]))

        da_var = ds_sub[nc_var]
        da_interp = interpolate_hybrid_to_plev(da_var, ds_sub, TARGET_PLEV)

        # 先处理 lon
        if "lon" in da_interp.dims:
            da_interp = da_interp.mean(dim="lon")  # Tmin 这里也应该先做 zonal mean

        # 再处理 lat
        if var_key in ["T", "O3"] and "lat" in da_interp.dims:
            w = np.cos(np.deg2rad(da_interp["lat"]))
            da_interp = da_interp.weighted(w).mean(dim="lat")

        elif var_key == "T_min" and "lat" in da_interp.dims:
            da_interp = da_interp.min(dim="lat")

        if var_key == "O3":
            da_interp = da_interp * 1e6  # mol/mol -> ppmv

        da_interp = da_interp.transpose("plev", "time")
        return da_interp.load()


def extract_hindcast_profile(member_id, var_key):
    folder = os.path.join(HINDCAST_BASE, VAR_INFO[var_key]["dir"])
    files = sorted(glob.glob(os.path.join(folder, "*.nc")))

    member_id = str(member_id).zfill(3)
    target_file = None
    for f in files:
        try:
            if parse_member_id(f) == member_id:
                target_file = f
                break
        except Exception:
            pass

    if target_file is None:
        return None

    return process_single_profile_file(
        target_file,
        var_key=var_key,
        start_month=START_MONTH,
        end_month=END_MONTH,
        drop_may_31=DROP_MAY_31,
        target_year=None,    # hindcast 成员一般是单年文件，不需要指定物理年
    )


def extract_ref_profile(var_key):
    ref_key = "T" if var_key == "T_min" else var_key
    file_path = REF_FILES[ref_key]

    return process_single_profile_file(
        file_path,
        var_key=var_key,
        start_month=START_MONTH,
        end_month=END_MONTH,
        drop_may_31=DROP_MAY_31,
        target_year=TARGET_YEAR,   # BWCN 多年文件时需要抽物理年
    )


# ============================================================
# 4. 1D 时间序列提取
# ============================================================
def get_u_series(file_path, target_year=None):
    da = process_single_profile_file(
        file_path,
        var_key="U",
        start_month=START_MONTH,
        end_month=END_MONTH,
        drop_may_31=DROP_MAY_31,
        target_year=target_year,
    )
    return da.sel(plev=U_PLEV_HPA, method="nearest").load()


def get_t_min_series(file_path, target_year=None):
    da = process_single_profile_file(
        file_path,
        var_key="T_min",
        start_month=START_MONTH,
        end_month=END_MONTH,
        drop_may_31=DROP_MAY_31,
        target_year=target_year,
    )
    return da.sel(plev=TMIN_PLEV_HPA, method="nearest").load()


def get_epflux_100hpa_series(file_path, target_year=None):
    da = process_single_profile_file(
        file_path,
        var_key="EPFLUX",
        start_month=START_MONTH,
        end_month=END_MONTH,
        drop_may_31=DROP_MAY_31,
        target_year=target_year,
    )
    da100 = da.sel(plev=EP_PLEV_HPA, method="nearest")
    return (-da100).load()   # 按你之前的逻辑，取负号


# ============================================================
# 5. O3 排序
# ============================================================
def rank_members_by_o3_rmse():
    print(f"\n[STEP 1] 计算 {CASE_NAME} 的 O3 RMSE 排序...")

    ref_o3 = get_o3_partial_column_series(
        REF_FILES["O3"],
        start_month=START_MONTH,
        end_month=END_MONTH,
        drop_may_31=DROP_MAY_31,
        target_year=TARGET_YEAR
    )

    hindcast_files = sorted(glob.glob(os.path.join(HINDCAST_BASE, "O3", "*.nc")))
    print(f"[INFO] 在 {HINDCAST_BASE}/O3 中找到 {len(hindcast_files)} 个成员文件。")

    results = []
    for f in hindcast_files:
        try:
            member_id = parse_member_id(f)
            h_o3 = get_o3_partial_column_series(
                f,
                start_month=START_MONTH,
                end_month=END_MONTH,
                drop_may_31=DROP_MAY_31,
                target_year=None
            )

            ref_aligned, h_aligned = xr.align(ref_o3, h_o3, join="inner")
            if ref_aligned.sizes["time"] == 0:
                raise ValueError("参考场与 Hindcast 在时间上没有重叠。")

            rmse = float(np.sqrt(np.mean((h_aligned.values - ref_aligned.values) ** 2)))

            results.append({
                "Member": member_id,
                "RMSE_DU": rmse,
                "File": f
            })
            print(f"  -> Member {member_id} | RMSE = {rmse:.3f} DU")

        except Exception as e:
            print(f"  [WARN] 排序阶段跳过 {os.path.basename(f)}: {e}")

    if len(results) == 0:
        raise RuntimeError("没有成功计算出任何成员的 O3 RMSE。")

    df = pd.DataFrame(results)
    df_sorted = df.sort_values(by="RMSE_DU", ascending=True).reset_index(drop=True)
    df_sorted.index += 1

    with open(TXT_OUT, "w") as f:
        f.write(f"{CASE_NAME} O3 Evaluation vs BWCN 0008\n")
        f.write(f"Metric: RMSE of 60–90N, 30–70 hPa Partial Column O3 (DU)\n")
        f.write(f"Period: month {START_MONTH:02d}-01 to {END_MONTH:02d}-30")
        if DROP_MAY_31:
            f.write(" (May 31 excluded)")
        f.write("\n")
        f.write("=" * 70 + "\n")
        f.write(f"{'Rank':<6} | {'Member':<10} | {'RMSE (DU)':<15}\n")
        f.write("-" * 40 + "\n")
        for rank, row in df_sorted.iterrows():
            f.write(f"{rank:<6} | {row['Member']:<10} | {row['RMSE_DU']:<15.4f}\n")

    df_sorted.to_csv(CSV_OUT, index_label="Rank")

    print(f"[SUCCESS] 排序结果已保存:")
    print(f"  TXT: {TXT_OUT}")
    print(f"  CSV: {CSV_OUT}")

    return df_sorted


# ============================================================
# 6. 绝对值剖面图：Best vs Worst
# ============================================================
def plot_best_vs_worst_absolute_profiles(df_sorted):
    print(f"\n[STEP 2] 绘制 Best vs Worst 绝对值剖面图...")

    top_n = min(COMPARE_N, len(df_sorted))
    best_members = df_sorted.head(top_n)["Member"].tolist()
    worst_members = df_sorted.tail(top_n)["Member"].tolist()

    print(f"[INFO] Best {top_n}:  {best_members}")
    print(f"[INFO] Worst {top_n}: {worst_members}")

    for var_key in ["U", "T", "O3", "EPFLUX"]:
        print(f"  [INFO] 处理变量: {var_key}")

        best_das = []
        best_ids = []
        for mid in best_members:
            da = extract_hindcast_profile(mid, var_key)
            if da is not None:
                best_das.append(da)
                best_ids.append(mid)

        worst_das = []
        worst_ids = []
        for mid in worst_members:
            da = extract_hindcast_profile(mid, var_key)
            if da is not None:
                worst_das.append(da)
                worst_ids.append(mid)

        if len(best_das) == 0 or len(worst_das) == 0:
            print(f"  [WARN] {var_key} 无法集齐有效数据，跳过。")
            continue

        da_best = concat_members(best_das, best_ids).mean(dim="member")
        da_worst = concat_members(worst_das, worst_ids).mean(dim="member")
        da_best, da_worst = xr.align(da_best, da_worst, join="inner")

        x = np.arange(da_best.sizes["time"])
        plev = da_best["plev"].values
        ticks, ticklabels = get_time_ticks(da_best["time"])

        z_all = np.concatenate([da_best.values.ravel(), da_worst.values.ravel()])
        z_all = z_all[np.isfinite(z_all)]
        if z_all.size == 0:
            print(f"  [WARN] {var_key} 全是 NaN，跳过。")
            continue

        cfg = PLOT_CFG_ABS[var_key]
        if cfg["sym"]:
            vmax = np.nanpercentile(np.abs(z_all), 98)
            if not np.isfinite(vmax) or vmax == 0:
                vmax = 1.0
            levels = np.linspace(-vmax, vmax, 21)
        else:
            vmin = np.nanpercentile(z_all, 2)
            vmax = np.nanpercentile(z_all, 98)
            if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
                vmin, vmax = np.nanmin(z_all), np.nanmax(z_all)
                if vmin == vmax:
                    vmin -= 1
                    vmax += 1
            levels = np.linspace(vmin, vmax, 21)

        fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharey=True, constrained_layout=True)
        titles = [f"Best {top_n} (Low O3 RMSE)", f"Worst {top_n} (High O3 RMSE)"]

        for ax, da, ttl in zip(axes, [da_best, da_worst], titles):
            z = da.values.copy()
            if var_key == "EPFLUX":
                z = -z

            cf = ax.contourf(x, plev, z, levels=levels, cmap=cfg["cmap"], extend="both")
            cs = ax.contour(x, plev, z, levels=10, colors="k", linewidths=0.5, alpha=0.6)
            ax.clabel(cs, inline=True, fontsize=6, fmt="%.1f")

            if cfg["sym"]:
                ax.contour(x, plev, z, levels=[0], colors="k", linewidths=1.1)

            ax.set_yscale("log")
            ax.invert_yaxis()
            ax.set_ylim(100, 1)
            ax.set_yticks([100, 50, 30, 10, 5, 1])
            ax.yaxis.set_major_formatter(ScalarFormatter())

            ax.set_xticks(ticks)
            ax.set_xticklabels(ticklabels)

            ax.set_title(ttl)
            ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)

        axes[0].set_ylabel("Pressure (hPa)")
        cbar = fig.colorbar(cf, ax=axes, pad=0.02, shrink=0.85)
        cbar.set_label(cfg["label"])
        fig.suptitle(f"{cfg['title']} ({CASE_NAME})", fontsize=14)

        out_png = os.path.join(PLOT_OUT_DIR, f"{CASE_NAME}_{var_key}_Best_vs_Worst.png")
        plt.savefig(out_png, dpi=300)
        print(f"    [SAVE] {out_png}")
        maybe_show_or_close()


# ============================================================
# 7. 偏差图：Best/Worst - Reference
# ============================================================
def plot_best_vs_worst_bias_profiles(df_sorted):
    print(f"\n[STEP 3] 绘制 Best/Worst 偏差图...")

    top_n = min(COMPARE_N, len(df_sorted))
    best_members = df_sorted.head(top_n)["Member"].tolist()
    worst_members = df_sorted.tail(top_n)["Member"].tolist()

    for var_key in ["U", "T", "O3", "EPFLUX"]:
        print(f"  [INFO] 处理变量: {var_key}")

        try:
            da_ref = extract_ref_profile(var_key)
        except Exception as e:
            print(f"  [WARN] 参考场 {var_key} 读取失败，跳过: {e}")
            continue

        best_das = []
        best_ids = []
        for mid in best_members:
            da = extract_hindcast_profile(mid, var_key)
            if da is not None:
                best_das.append(da)
                best_ids.append(mid)

        worst_das = []
        worst_ids = []
        for mid in worst_members:
            da = extract_hindcast_profile(mid, var_key)
            if da is not None:
                worst_das.append(da)
                worst_ids.append(mid)

        if len(best_das) == 0 or len(worst_das) == 0:
            print(f"  [WARN] {var_key} Hindcast 数据不足，跳过。")
            continue

        da_best_mean = concat_members(best_das, best_ids).mean(dim="member")
        da_worst_mean = concat_members(worst_das, worst_ids).mean(dim="member")

        da_best_mean, da_worst_mean, da_ref = xr.align(da_best_mean, da_worst_mean, da_ref, join="inner")

        da_best_diff = da_best_mean - da_ref
        da_worst_diff = da_worst_mean - da_ref

        # T_min overlay：用于 T 和 O3 图
        do_tmin_overlay = var_key in ["T", "O3"]
        if do_tmin_overlay:
            try:
                da_ref_tmin = extract_ref_profile("T_min")

                best_tmin_das = []
                best_tmin_ids = []
                for mid in best_members:
                    da = extract_hindcast_profile(mid, "T_min")
                    if da is not None:
                        best_tmin_das.append(da)
                        best_tmin_ids.append(mid)

                worst_tmin_das = []
                worst_tmin_ids = []
                for mid in worst_members:
                    da = extract_hindcast_profile(mid, "T_min")
                    if da is not None:
                        worst_tmin_das.append(da)
                        worst_tmin_ids.append(mid)

                da_best_tmin = concat_members(best_tmin_das, best_tmin_ids).mean(dim="member")
                da_worst_tmin = concat_members(worst_tmin_das, worst_tmin_ids).mean(dim="member")

                da_ref_tmin, da_best_tmin, da_worst_tmin, da_ref = xr.align(
                    da_ref_tmin, da_best_tmin, da_worst_tmin, da_ref, join="inner"
                )
                da_best_diff, da_worst_diff, da_ref = xr.align(
                    da_best_diff, da_worst_diff, da_ref, join="inner"
                )

            except Exception as e:
                print(f"    [WARN] Tmin overlay 失败，继续但不叠加: {e}")
                do_tmin_overlay = False

        x = np.arange(da_best_diff.sizes["time"])
        plev = da_best_diff["plev"].values
        ticks, ticklabels = get_time_ticks(da_best_diff["time"])

        z_all = np.concatenate([da_best_diff.values.ravel(), da_worst_diff.values.ravel()])
        z_all = z_all[np.isfinite(z_all)]
        if z_all.size == 0:
            print(f"  [WARN] {var_key} 偏差全是 NaN，跳过。")
            continue

        cfg = PLOT_CFG_DIFF[var_key]
        vmax = np.nanpercentile(np.abs(z_all), 98)
        if not np.isfinite(vmax) or vmax == 0:
            vmax = 1.0
        levels = np.linspace(-vmax, vmax, 21)

        fig, axes = plt.subplots(1, 2, figsize=(10, 4.6), sharey=True, constrained_layout=True)
        titles = [
            f"Best {top_n} Bias (Hindcast - Ref)",
            f"Worst {top_n} Bias (Hindcast - Ref)"
        ]

        for ax, da_diff, ttl, which in zip(
            axes,
            [da_best_diff, da_worst_diff],
            titles,
            ["best", "worst"]
        ):
            z = da_diff.values.copy()
            if var_key == "EPFLUX":
                z = -z

            cf = ax.contourf(x, plev, z, levels=levels, cmap=cfg["cmap"], extend="both")
            cs = ax.contour(x, plev, z, levels=10, colors="k", linewidths=0.5, alpha=0.6)
            ax.clabel(cs, inline=True, fontsize=6, fmt="%.1f")

            if do_tmin_overlay:
                ref_tmin = da_ref_tmin.values
                mean_tmin = da_best_tmin.values if which == "best" else da_worst_tmin.values

                t_levels = [100.0, TMIN_THRESH_K]

                # 参考场 Tmin < 197K
                ax.contourf(
                    x, plev, ref_tmin,
                    levels=t_levels,
                    hatches=["///"],
                    colors="none"
                )
                ax.contour(
                    x, plev, ref_tmin,
                    levels=[TMIN_THRESH_K],
                    colors="black",
                    linestyles="dashed",
                    linewidths=1.5,
                    alpha=0.85
                )

                # Hindcast 集合平均 Tmin < 197K
                ax.contourf(
                    x, plev, mean_tmin,
                    levels=t_levels,
                    hatches=["\\\\\\\\"],
                    colors="none"
                )
                ax.contour(
                    x, plev, mean_tmin,
                    levels=[TMIN_THRESH_K],
                    colors="black",
                    linestyles="solid",
                    linewidths=1.5,
                    alpha=0.85
                )

                ax.text(
                    0.02, 0.03, "Hatches: Polar Min T < 197 K",
                    transform=ax.transAxes,
                    fontsize=8,
                    fontweight="bold",
                    bbox=dict(facecolor="white", alpha=0.8, edgecolor="none", pad=2)
                )

            ax.set_yscale("log")
            ax.invert_yaxis()
            ax.set_ylim(100, 1)
            ax.set_yticks([100, 50, 30, 10, 5, 1])
            ax.yaxis.set_major_formatter(ScalarFormatter())

            ax.set_xticks(ticks)
            ax.set_xticklabels(ticklabels)

            ax.set_title(ttl)
            ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)

        axes[0].set_ylabel("Pressure (hPa)")
        cbar = fig.colorbar(cf, ax=axes, pad=0.02, shrink=0.85)
        cbar.set_label(cfg["label"])
        fig.suptitle(f"{cfg['title']} ({CASE_NAME})", fontsize=14)

        out_png = os.path.join(PLOT_OUT_DIR, f"{CASE_NAME}_{var_key}_Bias_Best{COMPARE_N}_vs_Worst{COMPARE_N}.png")
        plt.savefig(out_png, dpi=300)
        print(f"    [SAVE] {out_png}")
        maybe_show_or_close()


# ============================================================
# 8. 时间序列图
# ============================================================
def plot_variable_series(data_dict, ref_series, best_members, worst_members,
                         title, ylabel, save_name, hlines=None):
    """
    data_dict: {member_id: DataArray(time)}
    """
    if len(data_dict) == 0:
        print(f"[WARN] {title}: 没有有效成员数据。")
        return

    # 全集合
    all_ids = list(data_dict.keys())
    all_stack = concat_members([data_dict[mid] for mid in all_ids], all_ids)
    ens_mean = all_stack.mean(dim="member")
    ens_std = all_stack.std(dim="member")

    # Best
    best_valid = [mid for mid in best_members if mid in data_dict]
    best_stack = concat_members([data_dict[mid] for mid in best_valid], best_valid)
    best_mean = best_stack.mean(dim="member") if best_stack is not None else None

    # Worst
    worst_valid = [mid for mid in worst_members if mid in data_dict]
    worst_stack = concat_members([data_dict[mid] for mid in worst_valid], worst_valid)
    worst_mean = worst_stack.mean(dim="member") if worst_stack is not None else None

    to_align = [ens_mean, ens_std]
    if best_mean is not None:
        to_align.append(best_mean)
    if worst_mean is not None:
        to_align.append(worst_mean)
    if ref_series is not None:
        to_align.append(ref_series)

    aligned = xr.align(*to_align, join="inner")

    idx = 0
    ens_mean = aligned[idx]; idx += 1
    ens_std = aligned[idx]; idx += 1

    if best_mean is not None:
        best_mean = aligned[idx]; idx += 1
    if worst_mean is not None:
        worst_mean = aligned[idx]; idx += 1
    if ref_series is not None:
        ref_series = aligned[idx]

    x = np.arange(ens_mean.sizes["time"])
    ticks, ticklabels = get_time_ticks(ens_mean["time"])

    fig, ax = plt.subplots(figsize=(10, 5.5), constrained_layout=True)

    # 1) 全部成员细线
    cmap = plt.cm.get_cmap("tab20", len(all_ids))
    for i, mid in enumerate(all_ids):
        s = data_dict[mid].reindex(time=ens_mean.time)
        ax.plot(x, s.values, color=cmap(i), lw=0.6, alpha=0.35, zorder=2)

    # 2) 集合均值 ±1σ
    ax.fill_between(
        x,
        (ens_mean - ens_std).values,
        (ens_mean + ens_std).values,
        color="limegreen",
        alpha=0.2,
        zorder=3,
        label="Ens Mean ±1σ"
    )

    # 3) 三条平均线
    ax.plot(x, ens_mean.values, color="limegreen", lw=2.5, label="All Ens Mean", zorder=5)

    if best_mean is not None:
        ax.plot(x, best_mean.values, color="dodgerblue", lw=2.5, label=f"Best {COMPARE_N} Mean", zorder=6)

    if worst_mean is not None:
        ax.plot(x, worst_mean.values, color="crimson", lw=2.5, label=f"Worst {COMPARE_N} Mean", zorder=6)

    # 4) 参考线
    if ref_series is not None:
        ax.plot(x, ref_series.values, color="black", lw=3.0, label="BWCN Reference", zorder=10)

    # 5) 水平参考线
    if hlines:
        for hl in hlines:
            ax.axhline(
                hl["val"],
                color=hl["color"],
                lw=1.2,
                ls="--",
                label=hl["label"],
                zorder=4
            )

    ax.set_xlim(0, x[-1] if len(x) > 0 else 1)
    ax.set_xticks(ticks)
    ax.set_xticklabels(ticklabels)

    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=13)
    ax.grid(True, linestyle=":", alpha=0.6)
    ax.legend(loc="best", frameon=True, edgecolor="none", facecolor="white", framealpha=0.85, fontsize=9)

    out_path = os.path.join(PLOT_OUT_DIR, save_name)
    plt.savefig(out_path, dpi=300)
    print(f"    [SAVE] {out_path}")
    maybe_show_or_close()


def plot_all_line_series(df_sorted):
    print(f"\n[STEP 4] 绘制 1D 时间序列图...")

    top_n = min(COMPARE_N, len(df_sorted))
    best_members = df_sorted.head(top_n)["Member"].tolist()
    worst_members = df_sorted.tail(top_n)["Member"].tolist()

    # ---------------- U ----------------
    print("  [INFO] U @ 60N, 10 hPa")
    u_ref = get_u_series(REF_FILES["U"], target_year=TARGET_YEAR)
    u_files = sorted(glob.glob(os.path.join(HINDCAST_BASE, "U", "*.nc")))
    u_dict = {}
    for f in u_files:
        try:
            mid = parse_member_id(f)
            u_dict[mid] = get_u_series(f, target_year=None)
        except Exception as e:
            print(f"    [WARN] 跳过 {os.path.basename(f)}: {e}")

    plot_variable_series(
        data_dict=u_dict,
        ref_series=u_ref,
        best_members=best_members,
        worst_members=worst_members,
        title=f"{CASE_NAME}: Zonal Wind at 60°N, 10 hPa",
        ylabel="U Zonal Wind (m/s)",
        save_name=f"{CASE_NAME}_U_60N_10hPa_Line_{COMPARE_N}.png",
        hlines=[{"val": 0, "color": "gray", "label": "U = 0 (SSW)"}]
    )

    # ---------------- T_min ----------------
    print("  [INFO] Tmin @ 50 hPa")
    t_ref = get_t_min_series(REF_FILES["T"], target_year=TARGET_YEAR)
    t_files = sorted(glob.glob(os.path.join(HINDCAST_BASE, "T", "*.nc")))
    t_dict = {}
    for f in t_files:
        try:
            mid = parse_member_id(f)
            t_dict[mid] = get_t_min_series(f, target_year=None)
        except Exception as e:
            print(f"    [WARN] 跳过 {os.path.basename(f)}: {e}")

    plot_variable_series(
        data_dict=t_dict,
        ref_series=t_ref,
        best_members=best_members,
        worst_members=worst_members,
        title=f"{CASE_NAME}: Zonal-Mean Polar Minimum T (60–90°N, 50 hPa)",
        ylabel="Minimum Temperature (K)",
        save_name=f"{CASE_NAME}_Tmin_60-90N_50hPa_Line_{COMPARE_N}.png",
        hlines=[{"val": TMIN_THRESH_K, "color": "purple", "label": f"Cl Activation ({TMIN_THRESH_K:.0f} K)"}]
    )

    # ---------------- O3 partial column ----------------
    print("  [INFO] O3 partial column")
    o3_ref = get_o3_partial_column_series(
        REF_FILES["O3"],
        start_month=START_MONTH,
        end_month=END_MONTH,
        drop_may_31=DROP_MAY_31,
        target_year=TARGET_YEAR
    )
    o3_files = sorted(glob.glob(os.path.join(HINDCAST_BASE, "O3", "*.nc")))
    o3_dict = {}
    for f in o3_files:
        try:
            mid = parse_member_id(f)
            o3_dict[mid] = get_o3_partial_column_series(
                f,
                start_month=START_MONTH,
                end_month=END_MONTH,
                drop_may_31=DROP_MAY_31,
                target_year=None
            )
        except Exception as e:
            print(f"    [WARN] 跳过 {os.path.basename(f)}: {e}")

    plot_variable_series(
        data_dict=o3_dict,
        ref_series=o3_ref,
        best_members=best_members,
        worst_members=worst_members,
        title=f"{CASE_NAME}: Partial Column O3 (60–90°N, 30–70 hPa)",
        ylabel="Partial Column O3 (DU)",
        save_name=f"{CASE_NAME}_O3_Col_60-90N_Line_{COMPARE_N}.png",
        hlines=[]
    )

    # ---------------- EPFLUX ----------------
    print("  [INFO] EPFLUX @ 100 hPa")
    ep_ref = get_epflux_100hpa_series(REF_FILES["EPFLUX"], target_year=TARGET_YEAR)
    ep_files = sorted(glob.glob(os.path.join(HINDCAST_BASE, "EPflux_daily", "*.nc")))
    ep_dict = {}
    for f in ep_files:
        try:
            mid = parse_member_id(f)
            ep_dict[mid] = get_epflux_100hpa_series(f, target_year=None)
        except Exception as e:
            print(f"    [WARN] 跳过 {os.path.basename(f)}: {e}")

    plot_variable_series(
        data_dict=ep_dict,
        ref_series=ep_ref,
        best_members=best_members,
        worst_members=worst_members,
        title=f"{CASE_NAME}: EP Flux Fz at 100 hPa",
        ylabel="EP Flux Fz (native sign flipped)",
        save_name=f"{CASE_NAME}_EPFLUX_100hPa_Line_{COMPARE_N}.png",
        hlines=[{"val": 0, "color": "gray", "label": "Fz = 0"}]
    )


# ============================================================
# 9. 主程序
# ============================================================
def main():
    print("=" * 80)
    print(f"Running full hindcast evaluation suite for: {CASE_NAME}")
    print(f"Time range: {month_abbr[START_MONTH]} 1 to {month_abbr[END_MONTH]} 30")
    if DROP_MAY_31:
        print("Note: May 31 is excluded.")
    print("=" * 80)

    df_sorted = rank_members_by_o3_rmse()
    plot_best_vs_worst_absolute_profiles(df_sorted)
    plot_best_vs_worst_bias_profiles(df_sorted)
    plot_all_line_series(df_sorted)

    print("\n" + "=" * 80)
    print("[DONE] 全部分析完成。")
    print(f"排序输出: {RANK_OUT_DIR}")
    print(f"图像输出: {PLOT_OUT_DIR}")
    print("=" * 80)


if __name__ == "__main__":
    main()

In [ ]:
import os
import glob
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

# ============================================================
# 1. 基础配置与配对信息
# ============================================================
HINDCAST_BASE = "/mnt/soclim0/public_data/weiji/Hindcast"
PLOT_OUT = "/home/weiji/restart_exam/code/20260208HINDCAST_SUPPLY/plots_pairs"
CSV_OUT_DIR = "/home/weiji/restart_exam/code/20260208HINDCAST_SUPPLY/csv_evals"
os.makedirs(PLOT_OUT, exist_ok=True)
os.makedirs(CSV_OUT_DIR, exist_ok=True)

# 定义 8 组实验配对 (Coupled, NOCOUPL, 参考年份)
EXP_PAIRS = [
    ("0008-02", "0008-02_NOCOUPL", "0008"),
    ("0008-03", "0008-03_NOCOUPL", "0008"),
    ("0013-02", "0013-02_NOCOUPL", "0013"),
    ("0013-03", "0013-03_NOCOUPL", "0013"),
    ("0014-02", "0014-02_NOCOUPL", "0014"),
    ("0014-03", "0014-03_NOCOUPL", "0014"),
    ("0019-02", "0019-02_NOCOUPL", "0019"),
    ("0019-03", "0019-03_NOCOUPL", "0019")
]

# BWCN 动态路径模板
REF_FILES_TMPL = {
    "U": "/mnt/soclim0/public_data/weiji/BWCN/U/BWCN.cam.h3.{year}.U.nc",
    "T": "/mnt/soclim0/public_data/weiji/BWCN/T/BWCN.cam.h3.{year}.T.nc",
    "O3": "/mnt/soclim0/public_data/weiji/BWCN/O3/BWCN.cam.h3.{year}.O3.nc",
    "EPFLUX": "/mnt/soclim0/public_data/weiji/BWCN/EPflux_daily/EPFLUX_BWCN_Daily_24yr.nc"
}

VAR_INFO = {
    "U": {"dir": "U", "nc_var": "U"},
    "T": {"dir": "T", "nc_var": "T"},
    "T_min": {"dir": "T", "nc_var": "T"}, 
    "O3": {"dir": "O3", "nc_var": "O3"},
    "EPFLUX": {"dir": "EPflux_daily", "nc_var": "EPFLUX"} 
}

PLOT_CFG_DIFF = {
    "U": {"cmap": "RdBu_r", "label": "ΔU (m/s)", "title": "U Bias (60°N)", "sym": True},
    "T": {"cmap": "RdBu_r", "label": "ΔT (K)", "title": "Temp Bias (60-90°N)", "sym": True},
    "O3": {"cmap": "BrBG", "label": "ΔO$_3$ (ppmv)", "title": "Ozone Bias (60-90°N)", "sym": True},
    "EPFLUX": {"cmap": "PuOr_r", "label": "ΔFz", "title": "EP Flux Fz Bias", "sym": True},
}

TARGET_PLEV = np.logspace(np.log10(1.0), np.log10(100.0), 31)

mpl.rc("xtick", direction="out", labelsize=9)
mpl.rc("ytick", direction="out", labelsize=9)

# ============================================================
# 2. 通用时间截取与积分工具
# ============================================================
def get_time_mask(ds, target_year, start_month):
    """动态截取从 start_month 到 5月30日 的数据，匹配参考年"""
    years_in_ds = np.unique(ds.time.dt.year)
    mask = (ds.time.dt.year == int(target_year)) if len(years_in_ds) > 1 else xr.ones_like(ds.time, dtype=bool)
    mask = mask & (ds.time.dt.month >= start_month) & (ds.time.dt.month <= 5)
    mask = mask & ~((ds.time.dt.month == 5) & (ds.time.dt.day == 31))
    return mask

def calc_parc_o3_hybrid(ds_sub, p_top_hpa=30.0, p_bot_hpa=70.0):
    g, M_air, Na = 9.80665, 28.964 / 1000.0, 6.02214e23
    factor = Na / (g * M_air * 2.687e20)
    P0, PS = ds_sub['P0'], ds_sub['PS']
    P_interface = ds_sub['hyai'] * P0 + ds_sub['hybi'] * PS
    
    p_i = P_interface.isel(ilev=slice(0, -1)).rename({'ilev': 'lev'}).assign_coords(lev=ds_sub['lev'])
    p_ip1 = P_interface.isel(ilev=slice(1, None)).rename({'ilev': 'lev'}).assign_coords(lev=ds_sub['lev'])
    
    p_layer_top = xr.where(p_i < p_ip1, p_i, p_ip1)
    p_layer_bot = xr.where(p_i > p_ip1, p_i, p_ip1)
    pT, pB = p_top_hpa * 100.0, p_bot_hpa * 100.0
    upper = xr.where(p_layer_top > pT, p_layer_top, pT)
    lower = xr.where(p_layer_bot < pB, p_layer_bot, pB)
    
    overlap = xr.where(lower > upper, lower - upper, 0.0)
    return (ds_sub['O3'] * overlap * factor).sum(dim='lev')

# ============================================================
# 3. 第一部分：批量计算所有实验的 RMSE (O3) 并保存
# ============================================================
def calculate_and_save_rmse():
    print("="*60)
    print("[INFO] 开始执行阶段 1：批量计算 16 套实验的 O3 RMSE...")
    print("="*60)
    
    all_exps = []
    for cpl, nocpl, y in EXP_PAIRS:
        all_exps.extend([(cpl, y), (nocpl, y)])
        
    for exp_name, ref_year in all_exps:
        csv_out = os.path.join(CSV_OUT_DIR, f"{exp_name}_O3_RMSE_Ranking.csv")
        if os.path.exists(csv_out):
            print(f"[跳过] {exp_name} 的 RMSE 已存在。")
            continue
            
        start_month = int(exp_name.split("-")[1][:2])
        
        # 1. 读取参考文件 O3
        ref_file = REF_FILES_TMPL["O3"].format(year=ref_year)
        ds_ref = xr.open_dataset(ref_file)
        mask_ref = get_time_mask(ds_ref, ref_year, start_month)
        ds_ref_sub = ds_ref.isel(time=mask_ref)
        o3_col_ref = calc_parc_o3_hybrid(ds_ref_sub)
        o3_ref_polar = o3_col_ref.sel(lat=slice(60.0, 90.0)).weighted(np.cos(np.deg2rad(o3_col_ref.sel(lat=slice(60.0, 90.0)).lat))).mean(dim=["lat", "lon"]).compute()
        ds_ref.close()
        
        # 2. 读取 Hindcast 成员
        hind_files = glob.glob(os.path.join(HINDCAST_BASE, exp_name, "O3", "*.nc"))
        if not hind_files:
            print(f"[警告] 找不到实验 {exp_name} 的 O3 数据。")
            continue
            
        results = []
        for f in hind_files:
            mid = os.path.basename(f).split(".")[5]
            ds_h = xr.open_dataset(f)
            mask_h = get_time_mask(ds_h, ref_year, start_month)
            ds_h_sub = ds_h.isel(time=mask_h)
            o3_col_h = calc_parc_o3_hybrid(ds_h_sub)
            o3_h_polar = o3_col_h.sel(lat=slice(60.0, 90.0)).weighted(np.cos(np.deg2rad(o3_col_h.sel(lat=slice(60.0, 90.0)).lat))).mean(dim=["lat", "lon"]).compute()
            ds_h.close()
            
            min_len = min(len(o3_ref_polar), len(o3_h_polar))
            rmse = float(np.sqrt(np.mean((o3_h_polar.values[:min_len] - o3_ref_polar.values[:min_len])**2)))
            results.append({"Member": mid, "RMSE_DU": rmse, "File": f})
            
        df = pd.DataFrame(results).sort_values(by="RMSE_DU", ascending=True).reset_index(drop=True)
        df.index += 1
        df.to_csv(csv_out, index_label="Rank")
        print(f" -> 成功计算 {exp_name} (共 {len(hind_files)} 个成员), 保存至: {os.path.basename(csv_out)}")

# ============================================================
# 4. 剖面插值与绘图逻辑
# ============================================================
def extract_profile(file_path, var_key, target_year, start_month):
    vinfo = VAR_INFO[var_key]
    ds = xr.open_dataset(file_path)
    nc_var = "Fz" if var_key == "EPFLUX" and "Fz" in ds else vinfo["nc_var"]
    if nc_var not in ds:
        nc_var = list(ds.data_vars)[0]

    mask = get_time_mask(ds, target_year, start_month)
    ds_sub = ds.isel(time=mask)
    
    if var_key == "EPFLUX":
        da = ds_sub[nc_var]
        if "lev" in da.dims:
            da = da.rename({"lev": "plev"})
        if "plev" in da.dims:
            if da["plev"].max().values > 2000:
                da = da.assign_coords(plev=da["plev"] / 100.0)
            da = da.where((da.plev >= 1.0) & (da.plev <= 100.0), drop=True)
        res = da.transpose("plev", "time").compute()
        ds.close()
        return res

    if var_key == "U" and "lat" in ds_sub.dims:
        ds_sub = ds_sub.sel(lat=60.0, method="nearest")
    elif var_key in ["T", "T_min", "O3"] and "lat" in ds_sub.dims:
        ds_sub = ds_sub.sel(lat=slice(60.0, 90.0))

    P0 = ds_sub['P0'].values if 'P0' in ds_sub else 100000.0
    p_3d = (ds_sub['hyam'] * P0 + ds_sub['hybm'] * ds_sub['PS']) / 100.0
    da_var = ds_sub[nc_var]

    def _interp1d(v, p):
        m = np.isfinite(v) & np.isfinite(p) & (p > 0)
        if m.sum() < 2:
            return np.full(len(TARGET_PLEV), np.nan)
        idx = np.argsort(p[m])
        return np.interp(np.log(TARGET_PLEV), np.log(p[m][idx]), v[m][idx], left=np.nan, right=np.nan)

    da_interp = xr.apply_ufunc(
        _interp1d, da_var, p_3d,
        input_core_dims=[['lev'], ['lev']],
        output_core_dims=[['plev']],
        vectorize=True,
        output_dtypes=[float],
        dask="allowed"
    )
    da_interp = da_interp.assign_coords(plev=TARGET_PLEV)

    # 先 zonal mean
    if "lon" in da_interp.dims:
        da_interp = da_interp.mean(dim="lon")
        
    # 再做极区平均 / 极区最小值
    if var_key in ["T", "O3"] and "lat" in da_interp.dims:
        w = np.cos(np.deg2rad(da_interp["lat"]))
        da_interp = da_interp.weighted(w).mean(dim="lat")
    elif var_key == "T_min" and "lat" in da_interp.dims:
        da_interp = da_interp.min(dim="lat")
            
    if var_key == "O3":
        da_interp = da_interp * 1e6

    res = da_interp.transpose("plev", "time").compute()
    ds.close()
    return res

def plot_experiment_pairs():
    print("\n" + "="*60)
    print("[INFO] 开始执行阶段 2：绘制 Ensemble Mean Anomaly 剖面图...")
    print("="*60)

    for cpl_exp, nocpl_exp, ref_year in EXP_PAIRS:
        start_month = int(cpl_exp.split("-")[1])
        print(f"\n[处理配对] {cpl_exp} vs {nocpl_exp} (Ref: {ref_year})")
        
        for var_key in ["U", "T", "O3", "EPFLUX"]:
            # 1. 提取 Reference
            ref_path = REF_FILES_TMPL.get(var_key.replace("T_min", "T")).format(year=ref_year)
            da_ref = extract_profile(ref_path, var_key, ref_year, start_month)
            da_ref_tmin = extract_profile(REF_FILES_TMPL["T"].format(year=ref_year), "T_min", ref_year, start_month) if var_key in ["T", "O3"] else None

            # 2. 提取 Coupled 集合平均
            cpl_files = glob.glob(os.path.join(HINDCAST_BASE, cpl_exp, VAR_INFO[var_key]["dir"], "*.nc"))
            cpl_das = [extract_profile(f, var_key, ref_year, start_month) for f in cpl_files]
            if not cpl_das: continue
            da_cpl_mean = xr.concat(cpl_das, dim="member").mean(dim="member")
            
            cpl_das_tmin = [extract_profile(f.replace(VAR_INFO[var_key]["dir"], "T"), "T_min", ref_year, start_month) for f in glob.glob(os.path.join(HINDCAST_BASE, cpl_exp, "T", "*.nc"))] if var_key in ["T", "O3"] else []
            da_cpl_tmin_mean = xr.concat(cpl_das_tmin, dim="member").mean(dim="member") if cpl_das_tmin else None

            # 3. 提取 NOCOUPL 集合平均
            nocpl_files = glob.glob(os.path.join(HINDCAST_BASE, nocpl_exp, VAR_INFO[var_key]["dir"], "*.nc"))
            nocpl_das = [extract_profile(f, var_key, ref_year, start_month) for f in nocpl_files]
            if not nocpl_das: continue
            da_nocpl_mean = xr.concat(nocpl_das, dim="member").mean(dim="member")

            nocpl_das_tmin = [extract_profile(f.replace(VAR_INFO[var_key]["dir"], "T"), "T_min", ref_year, start_month) for f in glob.glob(os.path.join(HINDCAST_BASE, nocpl_exp, "T", "*.nc"))] if var_key in ["T", "O3"] else []
            da_nocpl_tmin_mean = xr.concat(nocpl_das_tmin, dim="member").mean(dim="member") if nocpl_das_tmin else None

            # 4. 计算 Anomaly (Ens Mean - Ref)
            min_len = min(da_cpl_mean.sizes["time"], da_ref.sizes["time"], da_nocpl_mean.sizes["time"])
            diff_cpl = da_cpl_mean.isel(time=slice(0, min_len)) - da_ref.isel(time=slice(0, min_len))
            diff_nocpl = da_nocpl_mean.isel(time=slice(0, min_len)) - da_ref.isel(time=slice(0, min_len))

            x_days = np.arange(min_len)
            plev = diff_cpl["plev"].values
            cfg = PLOT_CFG_DIFF[var_key]
            
            z_all = np.concatenate([diff_cpl.values.flatten(), diff_nocpl.values.flatten()])
            z_all = z_all[np.isfinite(z_all)]
            vmax = np.nanpercentile(np.abs(z_all), 98) if len(z_all) > 0 else 1.0
            if vmax == 0 or np.isnan(vmax): vmax = 1.0
            levels = np.linspace(-vmax, vmax, 21)

            # 5. 绘图
            fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharey=True, constrained_layout=True)
            titles = [f"{cpl_exp} Ens Mean Bias", f"{nocpl_exp} Ens Mean Bias"]
            diff_data = [diff_cpl, diff_nocpl]
            tmin_data = [da_cpl_tmin_mean, da_nocpl_tmin_mean]

            for i, (ax, diff_z, title) in enumerate(zip(axes, diff_data, titles)):
                z = diff_z.values
                if var_key == "EPFLUX": z = -z 
                
                cf = ax.contourf(x_days, plev, z, levels=levels, cmap=cfg["cmap"], extend="both")
                cs = ax.contour(x_days, plev, z, levels=10, colors="k", linewidths=0.5, alpha=0.6)
                ax.clabel(cs, inline=True, fontsize=6, fmt="%.1f")

                # T_min 197K Hatch 逻辑
                if var_key in ["T", "O3"] and da_ref_tmin is not None and tmin_data[i] is not None:
                    ref_z_min = da_ref_tmin.isel(time=slice(0, min_len)).values
                    mean_z_min = tmin_data[i].isel(time=slice(0, min_len)).values
                    t_levels = [100, 197]
                    
                    ax.contourf(x_days, plev, ref_z_min, levels=t_levels, hatches=['///'], colors='none')
                    ax.contour(x_days, plev, ref_z_min, levels=[197], colors='black', linestyles='dashed', linewidths=1.5, alpha=0.8)
                    
                    ax.contourf(x_days, plev, mean_z_min, levels=t_levels, hatches=['\\\\\\\\'], colors='none')
                    ax.contour(x_days, plev, mean_z_min, levels=[197], colors='black', linestyles='solid', linewidths=1.5, alpha=0.8)
                    
                    if i == 0:
                        ax.text(0.02, 0.03, "Hatches: Polar Min T < 197K\n(Dash: Ref, Solid: Hindcast)", 
                                transform=ax.transAxes, fontsize=8, fontweight='bold',
                                bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=2))

                ax.set_yscale("log")
                ax.invert_yaxis()
                ax.set_ylim(100, 1)
                ax.set_yticks([100, 50, 30, 10, 5, 1])
                ax.yaxis.set_major_formatter(ScalarFormatter())
                
                # 动态 X 轴刻度 (依赖启动月份)
                if start_month == 2:
                    ax.set_xticks([0, 28, 59, 89])
                    ax.set_xticklabels(["Feb 1", "Mar 1", "Apr 1", "May 1"])
                elif start_month == 3:
                    ax.set_xticks([0, 31, 61])
                    ax.set_xticklabels(["Mar 1", "Apr 1", "May 1"])
                    
                ax.set_title(title)
                ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)

            axes[0].set_ylabel("Pressure (hPa)")
            cbar = fig.colorbar(cf, ax=axes, pad=0.02, shrink=0.85)
            cbar.set_label(cfg["label"])
            
            fig.suptitle(f"{cfg['title']} Anomaly (Hindcast - BWCN {ref_year})", fontsize=14)
            out_png = os.path.join(PLOT_OUT, f"{var_key}_Bias_{cpl_exp}_vs_NOCOUPL.png")
            plt.savefig(out_png, dpi=300)
            plt.close() # 批量绘图时使用 close 释放内存

    print(f"\n[SUCCESS] 所有配对图像绘制完毕！图像保存在: {PLOT_OUT}")

if __name__ == "__main__":
    calculate_and_save_rmse()
    plot_experiment_pairs()

In [ ]:
"""
================================================================================
💡 备忘录 / To-Do:
之后准备标准化寻找 EP FLUX vs RMSE 的方法：
1. 方案 A：是否从 O3/U diverge（发生显著偏离）的时间点，往前推 10 天，
   再取 20 天的时间窗口来计算 EP Flux 的积分？
2. 方案 B：还是说设定当 EP FLUX 的差异达到某种物理阈值后，自动锁定/选择 
   EP Flux 产生强迫的时间窗口 (EPFLUX window)？

*后续需要通过分析刚刚跑出来的这 16 套数据（Coupled vs NOCOUPL）
 在各参考年份的表现，来量化这个“diverge point”的数学定义。
 最好关联上lowest O3 minimum的时间点。或者是diverge的时间点。
================================================================================
"""